In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import FinanceDataReader as fdr

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# =========================================================
# 기본 설정
# =========================================================

### 데이터 티커 및 기간 설정
ETF_CODE = "SMH"
START_DATE = "2020-01-01"
END_DATE = None

### Target 설정
N_DAYS = 5                                  # N일 후의 상승/하락 예측
THRESHOLD = 0.01                            # 상승/하락 판단 기준 (예: 0.05는 5% 상승/하락)
VALID_MONTHS = 1                            # 검증 데이터 기간 (개월 단위, 예: 1은 최근 1개월)


### 변수 선택시 
VIF_THRESHOLD = 10                          # 변수 선택시 VIF 기준 (예: 10 이상인 변수 제거)
LAG_SEARCH_YEARS = 1                        # 변수별 최적 lag 탐색시 사용할 최근 데이터 기간 (년 단위)
LAG_DAYS = [1, 3, 5, 10, 20, 40, 60, 120]   # 변수별 최적 lag 탐색시 사용할 일수 (예: 1, 3, 5, 10, 20, 40, 60, 120일)

### RF 설정 (변수 중요도 파악을 위한 과적합용 모델)
RANDOM_STATE = 42
N_RF_RUNS = 3
N_REPEATS = 10
TOP_N = 30 # 상위 N개 변수 선택
PRED_THRESHOLD = 0.5


# 외부 지표
EXTERNAL_TICKERS = {
    "QQQ": "QQQ",
    "SPY": "SPY",
    "SOXX": "SOXX",
    "NVDA": "NVDA",
    "TSM": "TSM",
    "VIX": "^VIX",
    "TNX": "^TNX",
    "USDKRW": "KRW=X",
    "DXY": "DX-Y.NYB",
    "GOLD": "GC=F",
    "OIL": "CL=F",
}

EXTERNAL_FEATURE_TYPES = {
    "QQQ": "price",
    "SPY": "price",
    "SOXX": "price",
    "NVDA": "price",
    "TSM": "price",
    "VIX": "risk",
    "TNX": "rate",
    "USDKRW": "price",
    "DXY": "price",
    "GOLD": "price",
    "OIL": "price",
}

model_configs = [
    {"model_name": "random_forest", "model_params": {"n_estimators": 500, "max_depth": None}},
    {"model_name": "extra_trees", "model_params": {"n_estimators": 500, "max_depth": None}},
    {"model_name": "gradient_boosting", "model_params": {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 3}},
    {"model_name": "hist_gradient_boosting", "model_params": {"max_iter": 300, "learning_rate": 0.03}},
    {"model_name": "logistic", "model_params": {"C": 1.0}},
    {"model_name": "svc", "model_params": {"C": 1.0, "kernel": "rbf"}},
]


from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline


In [1]:
import requests

verify_ssl=False
session = requests.Session()
session.verify = verify_ssl
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

original_get = requests.get

def custom_get(*args, **kwargs):
    kwargs["verify"] = verify_ssl
    return session.get(*args, **kwargs)

# FinanceDataReader 내부 requests.get 일시 덮어쓰기
requests.get = custom_get

In [3]:
etf_list = fdr.StockListing("ETF/KR")
etf_list.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,123350,2,290,0.24,123321.0,43.5661,13208639,1628702,267546
1,360750,4,TIGER 미국S&P500,28100,2,355,1.28,28010.0,14.0257,15775739,441821,184097
2,396500,2,TIGER 반도체TOP10,48000,5,-395,-0.82,48047.0,57.2504,10653376,512039,139968
3,102110,1,TIGER 200,123440,2,330,0.27,123372.0,43.7360,5164859,636988,107269
4,133690,4,TIGER 미국나스닥100,197990,2,2740,1.40,197210.0,24.3035,822145,162194,106083


In [4]:
## 함수

################################################################################################################################################
## 1. Base Feature Dataset 생성
################################################################################################################################################
def load_price_data(ticker, start_date="2020-01-01", end_date=None):
    """
    FinanceDataReader로 가격 데이터를 가져온다.
    Date 컬럼을 일반 컬럼으로 유지한다.
    """
    df = fdr.DataReader(ticker, start_date, end_date)
    df = df.reset_index().rename(columns={"index": "Date"})

    # 컬럼명 정리
    df["Date"] = pd.to_datetime(df["Date"])

    return df

def make_target_etf_features(etf_df, prefix):
    """
    예측 대상 ETF용 feature 생성.
    target은 여기서 만들지 않는다.
    """
    df = etf_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]
    volume = df["Volume"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    # target 만들 때 필요하므로 Close는 반드시 보존
    result[f"{prefix}_adj_close"] = close

    # 수익률
    result[f"{prefix}_ret_1d"] = close.pct_change(1)
    result[f"{prefix}_ret_5d"] = close.pct_change(5)
    result[f"{prefix}_ret_20d"] = close.pct_change(20)

    # 이동평균 대비 위치
    ma_5 = close.rolling(5).mean()
    ma_20 = close.rolling(20).mean()
    ma_60 = close.rolling(60).mean()

    result[f"{prefix}_ma5_ratio"] = close / ma_5 - 1
    result[f"{prefix}_ma20_ratio"] = close / ma_20 - 1
    result[f"{prefix}_ma60_ratio"] = close / ma_60 - 1

    # 변동성
    result[f"{prefix}_vol_20d"] = result[f"{prefix}_ret_1d"].rolling(20).std()

    # 거래량 비율
    vol_ma20 = volume.rolling(20).mean()
    result[f"{prefix}_volume_ratio_20d"] = volume / vol_ma20 - 1

    return result

def make_external_features(raw_df, name, feature_type="price"):
    """
    외부 지표용 최소 파생변수 생성.
    feature_type:
        - price: 일반 가격형 지표
        - risk: VIX 같은 리스크 레벨 지표
        - rate: 금리 지표
    """
    df = raw_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    if feature_type == "price":
        result[f"{name}_ret_5d"] = close.pct_change(5)
        result[f"{name}_ret_20d"] = close.pct_change(20)

    elif feature_type == "risk":
        result[f"{name}_level"] = close
        result[f"{name}_chg_5d"] = close.diff(5)
        result[f"{name}_chg_20d"] = close.diff(20)

    elif feature_type == "rate":
        result[f"{name}_level"] = close
        result[f"{name}_diff_5d"] = close.diff(5)
        result[f"{name}_diff_20d"] = close.diff(20)

    else:
        raise ValueError("feature_type must be one of ['price', 'risk', 'rate']")

    return result

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.
    """
    # 1. ETF 본체 로드
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 2. ETF 본체 feature 생성
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # 3. 외부 지표 붙이기
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # 4. 날짜 정렬
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    # 5. feature_cols 정리
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.

    주말 데이터가 섞여 NA가 늘어나는 문제를 막기 위해
    ETF / 외부 ticker 모두 영업일(월~금)만 사용한다.
    """

    # =========================================================
    # 0. 영업일 필터 함수
    # =========================================================
    def keep_weekdays_only(df, date_col="Date"):
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df[df[date_col].dt.weekday < 5]  # 월=0, 금=4
        df = df.sort_values(date_col).reset_index(drop=True)
        return df

    # =========================================================
    # 1. ETF 본체 로드
    # =========================================================
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 주말 제거
    etf_raw = keep_weekdays_only(etf_raw, date_col="Date")

    # =========================================================
    # 2. ETF 본체 feature 생성
    # =========================================================
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # feature 생성 후에도 혹시 모르니 다시 주말 제거
    base_df = keep_weekdays_only(base_df, date_col="Date")

    # =========================================================
    # 3. 외부 지표 붙이기
    # =========================================================
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            # 외부 ticker도 주말 제거
            raw = keep_weekdays_only(raw, date_col="Date")

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            # 외부 feature 생성 후에도 다시 주말 제거
            ext_feat = keep_weekdays_only(ext_feat, date_col="Date")

            # ETF 거래일 기준으로 붙임
            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # =========================================================
    # 4. 날짜 정렬 + 주말 최종 제거
    # =========================================================
    base_df = keep_weekdays_only(base_df, date_col="Date")
    
    # =========================================================
    # 4.1. 외부 지표 NA 보정
    # - ETF 본체는 건드리지 않고
    # - 외부 지표만 ffill
    # =========================================================
    etf_prefix = f"{etf_code}_"

    external_cols = [
        col for col in base_df.columns
        if col != "Date" and not col.startswith(etf_prefix)
    ]

    base_df[external_cols] = base_df[external_cols].ffill()

    # =========================================================
    # 5. feature_cols 정리
    # =========================================================
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

################################################################################################################################################
## 2. VIF 기반 불필요 칼럼 제거
################################################################################################################################################
def reduce_features_by_vif(
    df,
    feature_cols,
    vif_threshold=30.0,
    date_col="Date",
    verbose=True
):
    """
    VIF 기준으로 다중공선성이 높은 feature를 반복 제거한다.

    주의:
    - target 생성 전 단계에서 실행한다.
    - lag 생성 전 단계에서 실행한다.
    - Date, adj_close 등 보존 컬럼은 feature_cols에 넣지 않는 것을 전제로 한다.
    """

    # 1. 숫자형 feature만 사용
    numeric_feature_cols = [
        col for col in feature_cols
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    work_df = df[numeric_feature_cols].copy()

    # 2. inf 처리
    work_df = work_df.replace([np.inf, -np.inf], np.nan)

    # 3. VIF 계산용 결측 제거
    #    여기서는 VIF 계산에만 dropna를 쓰고,
    #    원본 df 자체를 줄이지는 않는다.
    vif_calc_df = work_df.dropna(axis=0).copy()

    print("VIF 계산 대상 row 수:", len(vif_calc_df))
    print("VIF 계산 대상 feature 수:", len(numeric_feature_cols))

    if len(vif_calc_df) == 0:
        raise ValueError("VIF 계산 가능한 데이터가 없습니다. 결측값을 확인하세요.")

    # 4. 상수 컬럼 제거
    nunique = vif_calc_df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if len(constant_cols) > 0:
        print("상수 컬럼 제거:", constant_cols)

    remaining_cols = [
        col for col in numeric_feature_cols
        if col not in constant_cols
    ]

    removed_records = []

    # 5. VIF 반복 제거
    while True:
        if len(remaining_cols) <= 1:
            break

        X = vif_calc_df[remaining_cols].copy()

        # 표준화
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        vif_values = []

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_scaled, i)
            except Exception:
                vif = np.inf

            vif_values.append({
                "feature": col,
                "vif": vif
            })

        vif_df = pd.DataFrame(vif_values).sort_values("vif", ascending=False)

        max_vif_row = vif_df.iloc[0]
        max_feature = max_vif_row["feature"]
        max_vif = max_vif_row["vif"]

        if verbose:
            print(f"현재 max VIF: {max_vif:.2f} / feature: {max_feature}")

        if max_vif <= vif_threshold:
            break

        # 가장 VIF 높은 컬럼 제거
        remaining_cols.remove(max_feature)

        removed_records.append({
            "removed_feature": max_feature,
            "vif": max_vif,
            "remaining_feature_count": len(remaining_cols)
        })

    removed_vif_df = pd.DataFrame(removed_records)

    # 6. 최종 VIF 테이블 계산
    final_vif_records = []

    if len(remaining_cols) > 1:
        X_final = vif_calc_df[remaining_cols].copy()

        scaler = StandardScaler()
        X_final_scaled = scaler.fit_transform(X_final)

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_final_scaled, i)
            except Exception:
                vif = np.inf

            final_vif_records.append({
                "feature": col,
                "vif": vif
            })

        final_vif_df = pd.DataFrame(final_vif_records).sort_values("vif", ascending=False)

    else:
        final_vif_df = pd.DataFrame({
            "feature": remaining_cols,
            "vif": [np.nan] * len(remaining_cols)
        })

    print()
    print("========== VIF 제거 결과 ==========")
    print("초기 feature 수:", len(numeric_feature_cols))
    print("상수 제거 feature 수:", len(constant_cols))
    print("VIF 제거 feature 수:", len(removed_records))
    print("최종 feature 수:", len(remaining_cols))
    print("===================================")

    return remaining_cols, removed_vif_df, final_vif_df


################################################################################################################################################
## 3. Target 변수 생성
################################################################################################################################################
def add_target_column(
    df,
    close_col,
    n_days=5,
    threshold=0.05,
    target_col=None
):
    """
    현재 시점 기준 n_days 뒤 수익률이 threshold 이상이면 1, 아니면 0인 target 생성.

    예:
    n_days=5, threshold=0.05
    → 5거래일 뒤 수익률이 +5% 이상이면 target=1
    """

    result = df.copy()
    result = result.sort_values("Date").reset_index(drop=True)

    if target_col is None:
        target_col = f"target_{n_days}d_up_{int(threshold * 100)}pct"

    # 미래 가격
    future_close = result[close_col].shift(-n_days)

    # 미래 수익률
    result[f"future_ret_{n_days}d"] = future_close / result[close_col] - 1

    # target 생성
    result[target_col] = np.where(
        result[f"future_ret_{n_days}d"] >= threshold,
        1,
        0
    )

    # 마지막 n_days개는 미래 가격이 없으므로 제거
    result.loc[result[f"future_ret_{n_days}d"].isna(), target_col] = np.nan

    return result, target_col


################################################################################################################################################
## 4. Lag 생성 후 변수별 최적 lag 탐색
################################################################################################################################################

def find_best_lag_by_feature(
    df,
    feature_cols,
    target_col,
    lag_days=[1, 3, 5, 10, 20],
    date_col="Date",
    method="corr"
):
    """
    각 feature별로 target과 가장 관계가 강한 선행 lag를 찾는다.

    lag 의미:
    - lag=1  : feature의 1거래일 전 값으로 오늘 target 설명
    - lag=5  : feature의 5거래일 전 값으로 오늘 target 설명
    - lag=20 : feature의 20거래일 전 값으로 오늘 target 설명

    즉, feature가 먼저 움직이고 나중에 target이 움직이는 구조만 본다.
    """

    records = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        for lag in lag_days:
            temp = df[[date_col, col, target_col]].copy()

            # 선행변수 구조
            temp[f"{col}_lag{lag}"] = temp[col].shift(lag)

            temp = temp[[f"{col}_lag{lag}", target_col]].replace(
                [np.inf, -np.inf],
                np.nan
            ).dropna()

            if len(temp) < 30:
                continue

            x = temp[f"{col}_lag{lag}"]
            y = temp[target_col]

            if x.nunique() <= 1:
                corr = np.nan
            else:
                corr = x.corr(y)

            records.append({
                "feature": col,
                "lag": lag,
                "corr": corr,
                "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
                "n_rows": len(temp)
            })

    lag_result_df = pd.DataFrame(records)

    if lag_result_df.empty:
        raise ValueError("lag 탐색 결과가 비어 있습니다. feature_cols 또는 target_col을 확인하세요.")

    # feature별 abs_corr가 가장 큰 lag 선택
    best_lag_df = (
        lag_result_df
        .sort_values(["feature", "abs_corr"], ascending=[True, False])
        .groupby("feature", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_lag_df = best_lag_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

    return lag_result_df, best_lag_df

def make_lagged_dataset_by_best_lag(
    df,
    best_lag_df,
    target_col,
    close_col,
    n_days=5,
    date_col="Date"
):
    """
    best_lag_df 기준으로 feature별 최적 lag를 적용한 최종 모델용 데이터셋 생성.
    """

    result = pd.DataFrame()
    result[date_col] = df[date_col]
    result[close_col] = df[close_col]

    # 확인용 미래수익률 보존
    future_ret_col = f"future_ret_{n_days}d"
    if future_ret_col in df.columns:
        result[future_ret_col] = df[future_ret_col]

    # target 보존
    result[target_col] = df[target_col]

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    # 결측/무한값 제거
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna().reset_index(drop=True)

    return result, lagged_feature_cols



################################################################################################################################################
## 5. RandomForest in-sample 학습 + permutation importance
################################################################################################################################################

def run_rf_permutation_importance_in_sample(
    lagged_df,
    feature_cols,
    target_col,
    date_col="Date",
    close_col=None,
    n_rf_runs=3,
    n_repeats=10,
    random_state=42
):
    """
    lagged_df 기준으로 RandomForestClassifier를 in-sample 학습한 뒤
    permutation importance를 반복 계산한다.

    핵심:
    - RF를 n_rf_runs번 학습
    - 각 RF마다 permutation을 n_repeats번 수행
    - feature별 importance raw 값을 전부 저장
    - 최종 importance_df는 총 n_rf_runs * n_repeats개의 raw importance 기준으로 계산

    예:
    n_rf_runs=3, n_repeats=10이면
    feature별 importance 값 30개를 기반으로 평균/표준편차/스코어 계산
    """

    df = lagged_df.copy()

    # =====================================================
    # 1. 모델 input / target 분리
    # =====================================================

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X = X.replace([np.inf, -np.inf], np.nan)

    model_df = pd.concat([X, y], axis=1).dropna().copy()

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()


    # =====================================================
    # 2. 여러 RF run + permutation raw importance 저장
    # =====================================================

    importance_records = []
    baseline_records = []

    for run in range(n_rf_runs):
        print()
        print(f"========== RF RUN {run + 1} / {n_rf_runs} ==========")

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state + run,
            n_jobs=-1
        )

        rf.fit(X, y)

        # =================================================
        # 3. in-sample 예측 성능 확인
        # =================================================

        pred = rf.predict(X)
        pred_proba = rf.predict_proba(X)[:, 1]

        acc = accuracy_score(y, pred)
        precision = precision_score(y, pred, zero_division=0)
        recall = recall_score(y, pred, zero_division=0)
        f1 = f1_score(y, pred, zero_division=0)
        loss = log_loss(y, pred_proba)

        cm = confusion_matrix(y, pred)

        print("accuracy :", round(acc, 4))
        print("precision:", round(precision, 4))
        print("recall   :", round(recall, 4))
        print("f1       :", round(f1, 4))
        print("log_loss :", round(loss, 4))

        baseline_records.append({
            "run": run + 1,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "log_loss": loss,
            "pred_1_count": int((pred == 1).sum()),
            "actual_1_count": int((y == 1).sum())
        })

        # =================================================
        # 4. permutation importance
        # =================================================

        perm = permutation_importance(
            rf,
            X,
            y,
            scoring="neg_log_loss",
            n_repeats=n_repeats,
            random_state=random_state + run,
            n_jobs=-1
        )

        # 핵심:
        # perm.importances shape = (n_features, n_repeats)
        # 여기서 반복별 raw importance를 전부 저장한다.
        for i, col in enumerate(feature_cols):
            for repeat_idx, importance_value in enumerate(perm.importances[i]):
                importance_records.append({
                    "run": run + 1,
                    "repeat": repeat_idx + 1,
                    "feature": col,
                    "importance": importance_value
                })

    # =====================================================
    # 5. 결과 정리
    # =====================================================

    raw_importance_df = pd.DataFrame(importance_records)
    baseline_df = pd.DataFrame(baseline_records)

    importance_df = (
        raw_importance_df
        .groupby("feature", as_index=False)
        .agg(
            importance_mean=("importance", "mean"),
            importance_std=("importance", "std"),
            importance_var=("importance", "var"),
            importance_min=("importance", "min"),
            importance_max=("importance", "max"),
            run_count=("run", "nunique"),
            repeat_count=("repeat", "count")
        )
    )

    # 안정성 점수
    # 평균 중요도는 높고, 30회 전체 기준 표준편차는 낮을수록 높게
    importance_df["importance_score"] = (
        importance_df["importance_mean"]
        - importance_df["importance_std"].fillna(0)
    )

    importance_df = importance_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    return importance_df, raw_importance_df, baseline_df

def split_lagged_feature_name(feature_name):
    """
    feature_lag20 형태의 컬럼명을 원본 feature와 lag로 분리한다.
    """

    if "_lag" not in feature_name:
        return feature_name, np.nan

    base_name = feature_name.rsplit("_lag", 1)[0]
    lag = feature_name.rsplit("_lag", 1)[1]

    try:
        lag = int(lag)
    except:
        lag = np.nan

    return base_name, lag



In [5]:

# =========================================================
# 실행부 함수화
# - 1) top_feature_df 생성 함수
# - 2) valid 검증/예측 함수
# =========================================================

def build_top_feature_df(
    etf_code,
    n_days,
    threshold,
    lag_search_years,
    random_state=42,
    n_rf_runs=3,
    n_repeats=10,
    top_n=30,
    start_date=START_DATE,
    end_date=END_DATE,
    valid_months=VALID_MONTHS,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    vif_threshold=VIF_THRESHOLD,
    lag_days=LAG_DAYS,
    verbose=True
):
    """
    base_df 생성 → train/valid 분리 → VIF 제거 → target 생성
    → lag 탐색 → lagged_df 생성 → RF permutation importance
    → top_feature_df 추출까지 한 번에 수행한다.

    Returns
    -------
    result : dict
        검증 함수에서 다시 필요한 객체들을 모두 담아서 반환한다.
        주요 key:
        - top_feature_df
        - base_df
        - train_base_df
        - valid_base_df
        - close_col
        - target_col
        - lagged_df
        - lagged_feature_cols
        - best_lag_df
        - importance_df
        - importance_with_lag_df
    """

    # =========================================================
    # 1. target 없는 base dataset 생성
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("1. Base feature dataset created.")

    base_df, base_feature_cols, close_col = make_base_feature_dataset(
        etf_code=etf_code,
        external_tickers=external_tickers,
        external_feature_types=external_feature_types,
        start_date=start_date,
        end_date=end_date
    )

    max_date = base_df["Date"].max()
    valid_start_date = max_date - pd.DateOffset(months=valid_months)

    train_base_df = base_df[base_df["Date"] < valid_start_date].copy()
    valid_base_df = base_df[base_df["Date"] >= valid_start_date].copy()

    if verbose:
        print("train_base_df shape:", train_base_df.shape)
        print("valid_base_df shape:", valid_base_df.shape)
        print("close_col:", close_col)
        print("=" * 50)

    # =========================================================
    # 2. VIF 기반 불필요 칼럼 제거
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("2. VIF filtering completed.")

    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=base_feature_cols,
        vif_threshold=vif_threshold,
        date_col="Date",
        verbose=verbose
    )

    keep_cols = ["Date", close_col] + vif_feature_cols
    vif_filtered_df = train_base_df[keep_cols].copy()

    if verbose:
        print("vif_filtered_df shape:", vif_filtered_df.shape)
        print("제거된 컬럼:")
        print(removed_vif_df)
        print("=" * 50)

    # =========================================================
    # 3. Target 변수 생성
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("3. Target column added.")

    target_df, target_col = add_target_column(
        df=vif_filtered_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold
    )

    # target 없는 마지막 n_days 행 제거
    target_df = target_df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    if verbose:
        print("target_col:", target_col)
        print("target_df shape after dropna:", target_df.shape)
        print("target 분포:")
        print(target_df[target_col].value_counts())
        print("target 비율:")
        print(target_df[target_col].value_counts(normalize=True))
        print("=" * 50)

    # =========================================================
    # 4. 최근 lag_search_years 기준으로 lag 탐색
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("4. 변수별 최적 LAG 탐색 완료.")

    max_date = target_df["Date"].max()
    lag_search_start_date = max_date - pd.DateOffset(years=lag_search_years)

    target_df_for_lag_search = target_df[
        target_df["Date"] >= lag_search_start_date
    ].copy()

    if verbose:
        print("lag 탐색 기준 기간:")
        print(target_df_for_lag_search["Date"].min(), "~", target_df_for_lag_search["Date"].max())
        print("lag 탐색용 데이터 shape:", target_df_for_lag_search.shape)

    exclude_cols_for_lag = [
        "Date",
        close_col,
        f"future_ret_{n_days}d",
        target_col
    ]

    lag_search_feature_cols = [
        col for col in target_df.columns
        if col not in exclude_cols_for_lag
    ]

    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=target_df_for_lag_search,
        feature_cols=lag_search_feature_cols,
        target_col=target_col,
        lag_days=lag_days,
        date_col="Date"
    )

    if verbose:
        print("전체 lag 탐색 결과 shape:", lag_result_df.shape)

    # =========================================================
    # 5. best lag 적용해서 lagged_df 생성
    # =========================================================
    lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date"
    )

    if verbose:
        print("lagged_df shape:", lagged_df.shape)
        print("lagged feature count:", len(lagged_feature_cols))
        print("=" * 50)

    # =========================================================
    # 6. permutation importance 실행
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("5. RandomForest in-sample 학습 + permutation importance 완료.")

    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=n_rf_runs,
        n_repeats=n_repeats,
        random_state=random_state
    )

    # =========================================================
    # 7. 결과 feature명 / lag 분리
    # =========================================================
    importance_view_df = importance_df.copy()

    importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    importance_view_df = importance_view_df[
        [
            "feature",
            "base_feature",
            "selected_lag",
            "importance_score",
            "importance_mean",
            "importance_std",
            "importance_var",
            "importance_min",
            "importance_max",
            "run_count",
            "repeat_count"
        ]
    ]

    # =========================================================
    # 8. best_lag_df와 importance 결과 합치기
    # =========================================================
    importance_with_lag_df = importance_view_df.merge(
        best_lag_df.rename(columns={
            "feature": "base_feature",
            "lag": "best_lag",
            "corr": "lag_corr",
            "abs_corr": "lag_abs_corr",
            "n_rows": "lag_n_rows"
        }),
        on="base_feature",
        how="left"
    )

    importance_with_lag_df = importance_with_lag_df[
        [
            "feature",
            "base_feature",
            "selected_lag",
            "importance_score",
            "importance_mean",
            "importance_std",
            "importance_var",
            "importance_min",
            "importance_max",
            "best_lag",
            "lag_corr",
            "lag_abs_corr",
            "lag_n_rows",
            "run_count",
            "repeat_count"
        ]
    ]

    importance_with_lag_df = importance_with_lag_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    # =========================================================
    # 9. TOP 변수 추출
    # =========================================================
    top_feature_df = importance_with_lag_df.head(top_n).copy()
    top_feature_cols = top_feature_df["feature"].tolist()

    if verbose:
        print("TOP feature count:", len(top_feature_cols))
        print(top_feature_cols)
        print("=" * 50)
        display(top_feature_df)

    result = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "lag_search_years": lag_search_years,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "start_date": start_date,
        "end_date": end_date,
        "valid_months": valid_months,
        "vif_threshold": vif_threshold,
        "lag_days": lag_days,

        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "train_base_df": train_base_df,
        "valid_base_df": valid_base_df,
        "close_col": close_col,

        "vif_feature_cols": vif_feature_cols,
        "removed_vif_df": removed_vif_df,
        "final_vif_df": final_vif_df,
        "vif_filtered_df": vif_filtered_df,

        "target_df": target_df,
        "target_col": target_col,

        "lag_result_df": lag_result_df,
        "best_lag_df": best_lag_df,
        "lagged_df": lagged_df,
        "lagged_feature_cols": lagged_feature_cols,

        "importance_df": importance_df,
        "raw_importance_df": raw_importance_df,
        "baseline_df": baseline_df,
        "importance_with_lag_df": importance_with_lag_df,

        "top_feature_df": top_feature_df,
        "top_feature_cols": top_feature_cols
    }

    return result



# =========================================================
# 검증용 분류 모델 생성 함수
# - top_feature_df를 뽑는 모델은 RF로 유지
# - valid 검증 단계에서만 model_name으로 모델을 바꿔 테스트
# =========================================================

def make_classifier_model(model_name="random_forest", random_state=42, model_params=None):
    """
    검증/예측 단계에서 사용할 분류모델을 생성한다.

    Parameters
    ----------
    model_name : str
        사용할 모델 이름.
        지원 모델:
        - "random_forest"
        - "extra_trees"
        - "gradient_boosting"
        - "hist_gradient_boosting"
        - "logistic"
        - "svc"
        - "knn"
    random_state : int
        random_state를 지원하는 모델에 적용.
    model_params : dict or None
        모델별 파라미터 덮어쓰기용 dict.

    Returns
    -------
    model : sklearn estimator
        fit / predict_proba 가능한 분류모델.
    """

    if model_params is None:
        model_params = {}

    model_name = model_name.lower()

    if model_name == "random_forest":
        default_params = dict(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )
        default_params.update(model_params)
        model = RandomForestClassifier(**default_params)

    elif model_name == "extra_trees":
        default_params = dict(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )
        default_params.update(model_params)
        model = ExtraTreesClassifier(**default_params)

    elif model_name == "gradient_boosting":
        default_params = dict(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=random_state
        )
        default_params.update(model_params)
        model = GradientBoostingClassifier(**default_params)

    elif model_name == "hist_gradient_boosting":
        default_params = dict(
            max_iter=300,
            learning_rate=0.03,
            max_leaf_nodes=31,
            random_state=random_state
        )
        default_params.update(model_params)
        model = HistGradientBoostingClassifier(**default_params)

    elif model_name == "logistic":
        default_params = dict(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=random_state
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(**default_params)
        )

    elif model_name == "svc":
        default_params = dict(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=random_state
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            SVC(**default_params)
        )

    elif model_name == "knn":
        default_params = dict(
            n_neighbors=15,
            weights="distance"
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(**default_params)
        )

    else:
        raise ValueError(
            "지원하지 않는 model_name입니다. "
            "사용 가능: random_forest, extra_trees, gradient_boosting, "
            "hist_gradient_boosting, logistic, svc, knn"
        )

    return model


def safe_binary_metrics(y_true, pred, pred_proba=None):
    """
    valid 구간에 한 클래스만 있는 경우에도 에러 없이 metric을 계산한다.
    """
    y_true = pd.Series(y_true).astype(int)
    pred = pd.Series(pred).astype(int)

    accuracy = accuracy_score(y_true, pred)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)

    auc = np.nan
    valid_logloss = np.nan

    if pred_proba is not None:
        pred_proba = np.asarray(pred_proba)
        if y_true.nunique() == 2:
            auc = roc_auc_score(y_true, pred_proba)
            valid_logloss = log_loss(y_true, pred_proba, labels=[0, 1])

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "logloss": valid_logloss,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def validate_top_feature_df(
    feature_result,
    model_name="random_forest",
    model_params=None,
    pred_threshold=0.5,
    random_state=42,
    n_days=None,
    threshold=None,
    verbose=True
):
    """
    build_top_feature_df() 결과를 받아서 valid_df 생성 후
    Train 전체 학습 → valid_df 예측 → precision 확인까지 수행한다.

    Parameters
    ----------
    feature_result : dict
        build_top_feature_df()에서 반환한 dict.
    model_name : str
        검증에 사용할 분류모델 이름.
    model_params : dict or None
        모델별 파라미터 덮어쓰기용 dict.
    pred_threshold : float
        예측 확률을 1로 바꿀 기준값.
    random_state : int
        RandomForest random_state.
    n_days : int or None
        None이면 feature_result의 n_days 사용.
    threshold : float or None
        None이면 feature_result의 threshold 사용.

    Returns
    -------
    result : dict
        - valid_df
        - valid_pred_df
        - eval_df
        - pred_1_df
        - metric_df
        - model
        - model_name
        - model_params
    """

    # =========================================================
    # 0. 필요한 객체 꺼내기
    # =========================================================
    top_feature_df = feature_result["top_feature_df"].copy()
    base_df = feature_result["base_df"].copy()
    valid_base_df = feature_result["valid_base_df"].copy()
    lagged_df = feature_result["lagged_df"].copy()
    close_col = feature_result["close_col"]
    target_col = feature_result["target_col"]

    if n_days is None:
        n_days = feature_result["n_days"]

    if threshold is None:
        threshold = feature_result["threshold"]

    # =========================================================
    # 1. valid_df 생성
    # =========================================================
    if verbose:
        print("=" * 60)
        print("6. valid_df 생성")
        print("=" * 60)

    if ("base_feature" not in top_feature_df.columns) or ("selected_lag" not in top_feature_df.columns):
        top_feature_df[["base_feature", "selected_lag"]] = top_feature_df["feature"].apply(
            lambda x: pd.Series(split_lagged_feature_name(x))
        )

    top_feature_df["selected_lag"] = top_feature_df["selected_lag"].astype(int)

    top_feature_cols = top_feature_df["feature"].tolist()
    top_base_features = top_feature_df["base_feature"].unique().tolist()

    if verbose:
        print("top feature 수:", len(top_feature_cols))

    need_cols = ["Date", close_col] + top_base_features
    missing_cols = [col for col in need_cols if col not in base_df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"base_df에 없는 컬럼이 있습니다: {missing_cols}")

    valid_df = base_df[need_cols].copy()
    valid_df = valid_df.sort_values("Date").reset_index(drop=True)

    # 전체 base_df 기준으로 lag 생성해야 최근 valid 구간의 lag가 계산됨
    for _, row in top_feature_df.iterrows():
        base_feature = row["base_feature"]
        selected_lag = int(row["selected_lag"])
        lagged_feature = row["feature"]

        valid_df[lagged_feature] = valid_df[base_feature].shift(selected_lag)

    if verbose:
        print("lag 생성 후 valid_df shape:", valid_df.shape)

    valid_df = valid_df[["Date", close_col] + top_feature_cols].copy()

    if verbose:
        print("원본 base_feature 제거 후 valid_df shape:", valid_df.shape)
        print("최종 컬럼 수:", len(valid_df.columns))

    valid_df, _ = add_target_column(
        df=valid_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold,
        target_col=target_col
    )

    if verbose:
        print("target 생성 후 valid_df shape:", valid_df.shape)

    valid_df = valid_df.tail(len(valid_base_df)).copy()
    valid_df = valid_df.reset_index(drop=True)

    if verbose:
        print("최종 valid_df shape:", valid_df.shape)
        print("valid_base_df shape:", valid_base_df.shape)
        print("valid_df 기간:")
        print(valid_df["Date"].min(), "~", valid_df["Date"].max())

    # =========================================================
    # 2. Train 학습 후 valid_df 예측
    # =========================================================
    if verbose:
        print("=" * 60)
        print("7. Train 학습 후 valid_df 예측")
        print("=" * 60)

    missing_train_cols = [col for col in top_feature_cols if col not in lagged_df.columns]
    missing_valid_cols = [col for col in top_feature_cols if col not in valid_df.columns]

    if len(missing_train_cols) > 0:
        raise ValueError(f"lagged_df에 없는 top feature가 있습니다: {missing_train_cols}")

    if len(missing_valid_cols) > 0:
        raise ValueError(f"valid_df에 없는 top feature가 있습니다: {missing_valid_cols}")

    if verbose:
        print("사용 feature 수:", len(top_feature_cols))

    train_df = lagged_df[
        ["Date", close_col, target_col] + top_feature_cols
    ].copy()

    train_df = train_df.replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=top_feature_cols + [target_col]).copy()

    X_train = train_df[top_feature_cols].copy()
    y_train = train_df[target_col].astype(int).copy()

    if verbose:
        print("train_df shape:", train_df.shape)
        print("X_train shape:", X_train.shape)
        print("train target 분포:")
        print(y_train.value_counts())
        print(y_train.value_counts(normalize=True))

    valid_pred_df = valid_df.copy()

    valid_pred_df["pred_proba"] = np.nan
    valid_pred_df["pred"] = np.nan

    valid_available_mask = (
        valid_pred_df[top_feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    X_valid = valid_pred_df.loc[valid_available_mask, top_feature_cols].copy()

    if verbose:
        print("valid_df shape:", valid_df.shape)
        print("예측 가능한 valid row 수:", len(X_valid))
        print("예측 불가능 row 수:", len(valid_df) - len(X_valid))

    model = make_classifier_model(
        model_name=model_name,
        random_state=random_state,
        model_params=model_params
    )

    model.fit(X_train, y_train)

    if verbose:
        print(f"모델 학습 완료: {model_name}")

    valid_pred_proba = model.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_pred_proba >= pred_threshold).astype(int)

    valid_pred_df.loc[valid_available_mask, "pred_proba"] = valid_pred_proba
    valid_pred_df.loc[valid_available_mask, "pred"] = valid_pred

    valid_pred_df["pred"] = valid_pred_df["pred"].astype("Int64")

    if verbose:
        print("valid_df 예측 완료")

    # =========================================================
    # 3. 예측 결과 확인
    # =========================================================
    display_cols = ["Date", close_col, "pred_proba", "pred"]

    if target_col in valid_pred_df.columns:
        display_cols = ["Date", close_col, target_col, "pred_proba", "pred"]

    if verbose:
        display(valid_pred_df[display_cols])
        print("예측값 분포:")
        print(valid_pred_df["pred"].value_counts(dropna=False))

    # =========================================================
    # 4. pred = 1 기준 precision 확인
    # =========================================================
    if verbose:
        print("=" * 60)
        print("8. 예측 1 기준 정확도 확인")
        print("=" * 60)

    eval_df = valid_pred_df[
        valid_pred_df["pred"].notna() &
        valid_pred_df[target_col].notna()
    ].copy()

    eval_df["pred"] = eval_df["pred"].astype(int)
    eval_df[target_col] = eval_df[target_col].astype(int)

    pred_1_df = eval_df[eval_df["pred"] == 1].copy()

    pred_1_count = len(pred_1_df)
    pred_1_actual_1_count = (pred_1_df[target_col] == 1).sum()

    if pred_1_count > 0:
        pred_1_precision = pred_1_actual_1_count / pred_1_count
    else:
        pred_1_precision = np.nan

    if len(eval_df) > 0:
        metric_dict = safe_binary_metrics(
            y_true=eval_df[target_col],
            pred=eval_df["pred"],
            pred_proba=eval_df["pred_proba"]
        )
    else:
        metric_dict = {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "logloss": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan,
        }

    metric_df = pd.DataFrame([{
        "etf_code": feature_result.get("etf_code"),
        "n_days": n_days,
        "threshold": threshold,
        "lag_search_years": feature_result.get("lag_search_years"),
        "top_n": feature_result.get("top_n"),
        "feature_count": len(top_feature_cols),
        "model_name": model_name,
        "model_params": str(model_params),
        "eval_count": len(eval_df),
        "pred_1_count": pred_1_count,
        "pred_1_actual_1_count": pred_1_actual_1_count,
        "pred_1_precision": pred_1_precision,
        "pred_threshold": pred_threshold,
        **metric_dict
    }])

    if verbose:
        print("평가 가능 row 수:", len(eval_df))
        print("예측 1 개수:", pred_1_count)
        print("예측 1 중 실제 1 개수:", pred_1_actual_1_count)
        print("예측 1 기준 정확도 precision:", pred_1_precision)
        display(metric_df)

    result = {
        "valid_df": valid_df,
        "valid_pred_df": valid_pred_df,
        "eval_df": eval_df,
        "pred_1_df": pred_1_df,
        "metric_df": metric_df,
        "model": model,
        "model_name": model_name,
        "model_params": model_params,
        "top_feature_cols": top_feature_cols
    }

    return result



def validate_multiple_models(
    feature_result,
    model_configs=None,
    pred_threshold=0.5,
    random_state=42,
    verbose=True
):
    """
    동일한 top_feature_df / valid_df 조건에서 여러 분류모델을 한 번에 검증한다.

    Parameters
    ----------
    feature_result : dict
        build_top_feature_df() 결과.
    model_configs : list[dict] or None
        예:
        [
            {"model_name": "random_forest", "model_params": {"n_estimators": 300}},
            {"model_name": "extra_trees", "model_params": {"n_estimators": 300}},
            {"model_name": "logistic", "model_params": {"C": 0.5}},
        ]
        None이면 기본 모델 세트를 사용한다.
    pred_threshold : float
        예측확률을 1로 바꿀 기준값.

    Returns
    -------
    result : dict
        - summary_df : 모델별 metric 비교표
        - results : 모델별 상세 결과 dict
    """

    if model_configs is None:
        model_configs = [
            {"model_name": "random_forest", "model_params": None},
            {"model_name": "extra_trees", "model_params": None},
            {"model_name": "gradient_boosting", "model_params": None},
            {"model_name": "hist_gradient_boosting", "model_params": None},
            {"model_name": "logistic", "model_params": None},
            {"model_name": "svc", "model_params": None},
        ]

    results = {}
    metric_list = []

    for cfg in model_configs:
        model_name = cfg.get("model_name")
        model_params = cfg.get("model_params")

        if verbose:
            print("=" * 80)
            print(f"모델 검증 시작: {model_name}")
            print("model_params:", model_params)
            print("=" * 80)

        result = validate_top_feature_df(
            feature_result=feature_result,
            model_name=model_name,
            model_params=model_params,
            pred_threshold=pred_threshold,
            random_state=random_state,
            verbose=verbose
        )

        results[model_name] = result
        metric_list.append(result["metric_df"])

    summary_df = pd.concat(metric_list, ignore_index=True)

    sort_cols = ["pred_1_precision", "precision", "recall", "f1"]
    summary_df = summary_df.sort_values(
        sort_cols,
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    if verbose:
        print("=" * 80)
        print("모델별 검증 결과 요약")
        print("=" * 80)
        display(summary_df)

    return {
        "summary_df": summary_df,
        "results": results
    }


In [6]:

# =========================================================
# Rolling 검증 + Excel 관리용 함수
# - base_df는 한 번만 만들고, fold별 as_of_date 기준으로 잘라서 사용
# - 하나의 파라미터 조합 + 하나의 모델 = 하나의 Excel run
# - run 내부에 rolling 평균/표준편차와 fold별 상세 결과 저장
# =========================================================

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime
import re


def get_nearest_trading_date_on_or_before(base_df, target_date, date_col="Date"):
    """target_date 이하에서 가장 가까운 실제 거래일을 반환한다."""
    dates = pd.to_datetime(base_df[date_col]).sort_values().dropna().unique()
    target_date = pd.to_datetime(target_date)
    valid_dates = dates[dates <= np.datetime64(target_date)]

    if len(valid_dates) == 0:
        raise ValueError(f"{target_date} 이전 거래일이 없습니다.")

    return pd.Timestamp(valid_dates[-1])


def make_rolling_as_of_dates(
    base_df,
    n_folds=10,
    step_months=1,
    end_date=None,
    date_col="Date"
):
    """
    최근 기준일부터 step_months씩 뒤로 밀면서 rolling 기준일 목록을 만든다.

    예:
    max_date=2026-05-20, n_folds=10, step_months=1
    → 2026-05-20, 2026-04-20, 2026-03-20 ... 근처의 실제 거래일
    """
    base_df = base_df.copy()
    base_df[date_col] = pd.to_datetime(base_df[date_col])

    if end_date is None:
        end_date = base_df[date_col].max()
    else:
        end_date = pd.to_datetime(end_date)

    as_of_dates = []

    for i in range(n_folds):
        raw_date = end_date - pd.DateOffset(months=i * step_months)
        as_of_date = get_nearest_trading_date_on_or_before(
            base_df=base_df,
            target_date=raw_date,
            date_col=date_col
        )
        as_of_dates.append(as_of_date)

    # 중복 제거. 휴장/월말 차이 때문에 드물게 중복될 수 있음.
    as_of_dates = list(dict.fromkeys(as_of_dates))
    return as_of_dates


def build_top_feature_df(
    etf_code,
    n_days,
    threshold,
    lag_search_years,
    random_state=42,
    n_rf_runs=3,
    n_repeats=10,
    top_n=30,
    start_date=START_DATE,
    end_date=END_DATE,
    valid_months=VALID_MONTHS,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    vif_threshold=VIF_THRESHOLD,
    lag_days=LAG_DAYS,
    base_df=None,
    base_feature_cols=None,
    close_col=None,
    as_of_date=None,
    verbose=True
):
    """
    base_df 생성/입력 → as_of_date 기준 자르기 → train/valid 분리 → VIF 제거 → target 생성
    → lag 탐색 → lagged_df 생성 → RF permutation importance → top_feature_df 추출.

    rolling 검증에서는 base_df/base_feature_cols/close_col을 미리 만들어 넣는 것을 권장한다.
    """

    # =========================================================
    # 1. target 없는 base dataset 준비
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("1. Base feature dataset prepared.")

    if base_df is None:
        base_df, base_feature_cols, close_col = make_base_feature_dataset(
            etf_code=etf_code,
            external_tickers=external_tickers,
            external_feature_types=external_feature_types,
            start_date=start_date,
            end_date=end_date
        )
    else:
        base_df = base_df.copy()
        if base_feature_cols is None:
            if close_col is None:
                close_col = f"{etf_code}_adj_close"
            base_feature_cols = [c for c in base_df.columns if c not in ["Date", close_col]]
        if close_col is None:
            close_col = f"{etf_code}_adj_close"

    base_df["Date"] = pd.to_datetime(base_df["Date"])
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    if as_of_date is not None:
        as_of_date = get_nearest_trading_date_on_or_before(base_df, as_of_date, date_col="Date")
        base_df = base_df[base_df["Date"] <= as_of_date].copy().reset_index(drop=True)

    max_date = base_df["Date"].max()
    valid_start_date = max_date - pd.DateOffset(months=valid_months)

    train_base_df = base_df[base_df["Date"] < valid_start_date].copy()
    valid_base_df = base_df[base_df["Date"] >= valid_start_date].copy()

    if len(train_base_df) == 0 or len(valid_base_df) == 0:
        raise ValueError("train_base_df 또는 valid_base_df가 비어 있습니다. 기간 설정을 확인하세요.")

    if verbose:
        print("as_of_date:", max_date)
        print("valid_start_date:", valid_start_date)
        print("train_base_df shape:", train_base_df.shape)
        print("valid_base_df shape:", valid_base_df.shape)
        print("close_col:", close_col)
        print("=" * 50)

    # =========================================================
    # 2. VIF 기반 불필요 컬럼 제거: train 구간만 사용
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("2. VIF filtering")

    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=base_feature_cols,
        vif_threshold=vif_threshold,
        date_col="Date",
        verbose=verbose
    )

    keep_cols = ["Date", close_col] + vif_feature_cols
    vif_filtered_df = train_base_df[keep_cols].copy()

    if verbose:
        print("vif_filtered_df shape:", vif_filtered_df.shape)
        print("=" * 50)

    # =========================================================
    # 3. Target 변수 생성: train 구간만 사용
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("3. Target column")

    target_df, target_col = add_target_column(
        df=vif_filtered_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold
    )

    target_df = target_df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    if target_df[target_col].nunique() < 2:
        raise ValueError(
            f"train target이 한 클래스만 존재합니다. target 분포: {target_df[target_col].value_counts().to_dict()}"
        )

    if verbose:
        print("target_col:", target_col)
        print("target_df shape after dropna:", target_df.shape)
        print("target 분포:")
        print(target_df[target_col].value_counts())
        print("=" * 50)

    # =========================================================
    # 4. 최근 lag_search_years 기준으로 lag 탐색: train 구간 내부만 사용
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("4. Best lag search")

    train_max_date = target_df["Date"].max()
    lag_search_start_date = train_max_date - pd.DateOffset(years=lag_search_years)

    target_df_for_lag_search = target_df[target_df["Date"] >= lag_search_start_date].copy()

    exclude_cols_for_lag = ["Date", close_col, f"future_ret_{n_days}d", target_col]
    lag_search_feature_cols = [col for col in target_df.columns if col not in exclude_cols_for_lag]

    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=target_df_for_lag_search,
        feature_cols=lag_search_feature_cols,
        target_col=target_col,
        lag_days=lag_days,
        date_col="Date"
    )

    # =========================================================
    # 5. best lag 적용해서 lagged_df 생성
    # =========================================================
    lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date"
    )

    if lagged_df[target_col].nunique() < 2:
        raise ValueError(
            f"lagged_df target이 한 클래스만 존재합니다. target 분포: {lagged_df[target_col].value_counts().to_dict()}"
        )

    if verbose:
        print("lagged_df shape:", lagged_df.shape)
        print("lagged feature count:", len(lagged_feature_cols))
        print("=" * 50)

    # =========================================================
    # 6. permutation importance 실행
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("5. RF permutation importance")

    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=n_rf_runs,
        n_repeats=n_repeats,
        random_state=random_state
    )

    # =========================================================
    # 7. 결과 feature명 / lag 분리 + best_lag_df 병합
    # =========================================================
    importance_view_df = importance_df.copy()
    importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    importance_view_df = importance_view_df[
        [
            "feature", "base_feature", "selected_lag",
            "importance_score", "importance_mean", "importance_std", "importance_var",
            "importance_min", "importance_max", "run_count", "repeat_count"
        ]
    ]

    importance_with_lag_df = importance_view_df.merge(
        best_lag_df.rename(columns={
            "feature": "base_feature",
            "lag": "best_lag",
            "corr": "lag_corr",
            "abs_corr": "lag_abs_corr",
            "n_rows": "lag_n_rows"
        }),
        on="base_feature",
        how="left"
    )

    importance_with_lag_df = importance_with_lag_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    top_feature_df = importance_with_lag_df.head(top_n).copy()
    top_feature_cols = top_feature_df["feature"].tolist()

    if verbose:
        print("TOP feature count:", len(top_feature_cols))
        print(top_feature_cols)
        print("=" * 50)
        display(top_feature_df)

    result = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "lag_search_years": lag_search_years,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "start_date": start_date,
        "end_date": end_date,
        "valid_months": valid_months,
        "vif_threshold": vif_threshold,
        "lag_days": lag_days,
        "as_of_date": max_date,
        "valid_start_date": valid_start_date,

        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "train_base_df": train_base_df,
        "valid_base_df": valid_base_df,
        "close_col": close_col,

        "vif_feature_cols": vif_feature_cols,
        "removed_vif_df": removed_vif_df,
        "final_vif_df": final_vif_df,
        "vif_filtered_df": vif_filtered_df,

        "target_df": target_df,
        "target_col": target_col,

        "lag_result_df": lag_result_df,
        "best_lag_df": best_lag_df,
        "lagged_df": lagged_df,
        "lagged_feature_cols": lagged_feature_cols,

        "importance_df": importance_df,
        "raw_importance_df": raw_importance_df,
        "baseline_df": baseline_df,
        "importance_with_lag_df": importance_with_lag_df,

        "top_feature_df": top_feature_df,
        "top_feature_cols": top_feature_cols
    }

    return result


def summarize_selected_features(fold_feature_results):
    """rolling fold별 top_feature_df를 모아서 반복 선택된 feature를 집계한다."""
    records = []

    for fold_result in fold_feature_results:
        fold = fold_result["fold"]
        as_of_date = fold_result["as_of_date"]
        top_df = fold_result["feature_result"]["top_feature_df"].copy()
        top_df["fold"] = fold
        top_df["as_of_date"] = as_of_date
        records.append(top_df)

    if len(records) == 0:
        return pd.DataFrame(), pd.DataFrame()

    all_selected_df = pd.concat(records, ignore_index=True)

    selected_summary_df = (
        all_selected_df
        .groupby("feature", as_index=False)
        .agg(
            selected_count=("fold", "nunique"),
            avg_importance_score=("importance_score", "mean"),
            avg_importance_mean=("importance_mean", "mean"),
            avg_importance_std=("importance_std", "mean"),
            avg_selected_lag=("selected_lag", "mean"),
            first_base_feature=("base_feature", "first")
        )
        .sort_values(["selected_count", "avg_importance_score"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return all_selected_df, selected_summary_df


def summarize_rolling_metrics(fold_metric_df):
    """fold별 metric을 평균/표준편차 metric dict로 변환한다."""
    metric_cols = [
        "accuracy", "precision", "recall", "f1", "auc", "logloss",
        "pred_1_precision", "eval_count", "pred_1_count",
        "pred_1_actual_1_count", "tp", "fp", "tn", "fn"
    ]

    summary = {}
    for col in metric_cols:
        if col not in fold_metric_df.columns:
            continue

        s = pd.to_numeric(fold_metric_df[col], errors="coerce")
        summary[f"rolling_{col}_mean"] = s.mean(skipna=True)
        summary[f"rolling_{col}_std"] = s.std(skipna=True)
        summary[f"rolling_{col}_min"] = s.min(skipna=True)
        summary[f"rolling_{col}_max"] = s.max(skipna=True)

    # pred_1_precision은 pred=1이 하나도 없으면 NaN이 되므로, 보수적으로 0 처리한 평균도 같이 저장
    if "pred_1_precision" in fold_metric_df.columns:
        s0 = pd.to_numeric(fold_metric_df["pred_1_precision"], errors="coerce").fillna(0)
        summary["rolling_pred_1_precision_zero_fill_mean"] = s0.mean()

    return summary




# =========================================================
# Excel 저장 유틸
# =========================================================
EXCEL_EXPERIMENT_DIR = Path("experiments_excel")


def make_safe_filename(value):
    """파일/폴더명에 쓰기 어려운 문자를 정리한다."""
    value = str(value)
    value = re.sub(r"[^0-9A-Za-z가-힣_.\-]+", "_", value)
    value = value.strip("_")
    return value[:120] if len(value) > 120 else value


def make_run_id(etf_code, model_name, n_days, threshold, top_n, run_name=None):
    """실험별 고유 run_id 생성."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if run_name is None:
        run_name = f"{etf_code}_{model_name}_{n_days}d_thr{threshold}_top{top_n}"

    return make_safe_filename(f"{timestamp}_{run_name}")


# ---------------------------------------------------------
# 중복 실험 스킵용 유틸
# ---------------------------------------------------------
EXPERIMENT_KEY_COLS = [
    "etf_code",
    "n_days",
    "threshold",
    "valid_months",
    "vif_threshold",
    "lag_search_years",
    "lag_days",
    "random_state",
    "n_rf_runs",
    "n_repeats",
    "top_n",
    "pred_threshold",
    "model_name",
    "model_params",
    "n_folds",
    "step_months",
    "start_date",
    "end_date",
]


SUMMARY_COLUMNS = [
    "experiment_key",
    "created_dt",
    "run_id",
    "etf_code",
    "rolling_precision_mean",
    "n_days",
    "threshold",
    "valid_months",
    "vif_threshold",
    "lag_search_years",
    "lag_days",
    "top_n",
    "pred_threshold",
    "model_name",
    "model_params",
    "n_folds",
    "step_months",
    "rolling_accuracy_mean",
    "rolling_accuracy_std",
    "rolling_accuracy_min",
    "rolling_accuracy_max",
    "rolling_precision_std",
    "rolling_precision_min",
    "rolling_precision_max",
    "rolling_recall_mean",
    "rolling_recall_std",
    "rolling_recall_min",
    "rolling_recall_max",
    "rolling_f1_mean",
    "rolling_f1_std",
    "rolling_f1_min",
    "rolling_f1_max",
    "rolling_auc_mean",
    "rolling_auc_std",
    "rolling_auc_min",
    "rolling_auc_max",
    "rolling_logloss_mean",
    "rolling_logloss_std",
    "rolling_logloss_min",
    "rolling_logloss_max",
    "n_folds_success",
    "error_count",
]


def normalize_experiment_value(value):
    """실험 조건 비교를 위해 값 표현을 안정적으로 통일한다."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return str(pd.to_datetime(value).date())
    if isinstance(value, float):
        return round(value, 12)
    if isinstance(value, dict):
        return {str(k): normalize_experiment_value(v) for k, v in sorted(value.items(), key=lambda x: str(x[0]))}
    if isinstance(value, (list, tuple, set)):
        return [normalize_experiment_value(v) for v in list(value)]
    if pd.isna(value) if not isinstance(value, (list, tuple, dict, set)) else False:
        return None
    return value


def make_experiment_key(config):
    """동일 실험 여부를 판단하는 key를 만든다. run_id처럼 시간은 포함하지 않는다."""
    key_dict = {
        col: normalize_experiment_value(config.get(col))
        for col in EXPERIMENT_KEY_COLS
    }
    key_json = json.dumps(key_dict, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.md5(key_json.encode("utf-8")).hexdigest()


def load_master_summary(experiment_dir=EXCEL_EXPERIMENT_DIR):
    """전체 summary Excel을 읽는다. 없으면 빈 DataFrame 반환."""
    master_summary_path = Path(experiment_dir) / "experiment_summary.xlsx"
    if not master_summary_path.exists():
        return pd.DataFrame()

    try:
        return pd.read_excel(master_summary_path, sheet_name="summary")
    except Exception:
        return pd.DataFrame()


def find_existing_experiment(config, experiment_dir=EXCEL_EXPERIMENT_DIR):
    """
    같은 실험 조건이 이미 저장되어 있는지 찾는다.
    - 새 버전 summary에는 experiment_key가 있으므로 experiment_key로 비교
    - 예전 summary에 experiment_key가 없으면 주요 config 컬럼 값으로 비교
    """
    master_summary_df = load_master_summary(experiment_dir)
    if master_summary_df.empty:
        return None

    target_key = make_experiment_key(config)

    for key_col in ["experiment_key", "run_key"]:
        if key_col in master_summary_df.columns:
            matched = master_summary_df[master_summary_df[key_col].astype(str) == str(target_key)]
            if len(matched) > 0:
                return matched.iloc[0].to_dict()

    compare_cols = [c for c in EXPERIMENT_KEY_COLS if c in master_summary_df.columns]
    if len(compare_cols) == 0:
        return None

    target_norm = {c: normalize_experiment_value(config.get(c)) for c in compare_cols}

    for _, row in master_summary_df.iterrows():
        is_same = True
        for col in compare_cols:
            old_value = row.get(col)
            # Excel에서 list/dict는 문자열로 저장되므로 문자열 비교까지 허용
            old_norm = normalize_experiment_value(old_value)
            new_norm = target_norm[col]
            if str(old_norm) != str(new_norm):
                is_same = False
                break
        if is_same:
            return row.to_dict()

    return None


def to_excel_safe_value(value):
    """dict/list/Timestamp 등을 Excel에 넣기 쉬운 값으로 변환한다."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return str(pd.to_datetime(value))
    if isinstance(value, Path):
        return str(value)
    return value


def make_config_df(config):
    """config dict를 key/value 형태의 DataFrame으로 변환한다."""
    return pd.DataFrame([
        {"key": k, "value": to_excel_safe_value(v)}
        for k, v in config.items()
    ])


def make_one_row_df(row_dict):
    """summary dict를 한 줄 DataFrame으로 변환한다."""
    return pd.DataFrame([{k: to_excel_safe_value(v) for k, v in row_dict.items()}])


def prepare_df_for_excel(df):
    """Excel 저장 전 object 컬럼 내 dict/list 등을 문자열로 변환한다."""
    if df is None:
        return pd.DataFrame()

    out = df.copy()

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].apply(to_excel_safe_value)

    return out


def write_excel_book(path, sheet_dict):
    """여러 DataFrame을 하나의 Excel 파일로 저장한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name, df in sheet_dict.items():
            safe_sheet_name = str(sheet_name)[:31]
            safe_df = prepare_df_for_excel(df)
            safe_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

            # 컬럼 너비 자동 보정
            ws = writer.sheets[safe_sheet_name]
            for col_cells in ws.columns:
                col_letter = col_cells[0].column_letter
                max_len = 0
                for cell in col_cells[:200]:
                    if cell.value is not None:
                        max_len = max(max_len, len(str(cell.value)))
                ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)

    return path


def concat_fold_validation_df(fold_validation_results, key):
    """fold_validation_results 안의 valid_pred_df/eval_df 등을 fold 정보와 함께 합친다."""
    records = []

    for item in fold_validation_results:
        fold = item["fold"]
        as_of_date = item["as_of_date"]
        valid_result = item["valid_result"]

        if key not in valid_result:
            continue

        df = valid_result[key]
        if df is None or len(df) == 0:
            continue

        tmp = df.copy()
        tmp.insert(0, "fold", fold)
        tmp.insert(1, "as_of_date", as_of_date)
        records.append(tmp)

    if len(records) == 0:
        return pd.DataFrame()

    return pd.concat(records, ignore_index=True)


def concat_fold_feature_df(fold_feature_results, key):
    """fold_feature_results 안의 top_feature_df/best_lag_df 등을 fold 정보와 함께 합친다."""
    records = []

    for item in fold_feature_results:
        fold = item["fold"]
        as_of_date = item["as_of_date"]
        feature_result = item["feature_result"]

        if key not in feature_result:
            continue

        df = feature_result[key]
        if df is None or len(df) == 0:
            continue

        tmp = df.copy()
        tmp.insert(0, "fold", fold)
        tmp.insert(1, "as_of_date", as_of_date)
        records.append(tmp)

    if len(records) == 0:
        return pd.DataFrame()

    return pd.concat(records, ignore_index=True)


def order_summary_df(summary_df, summary_columns=SUMMARY_COLUMNS):
    """summary 시트 컬럼 순서를 고정한다. 지정하지 않은 추가 컬럼은 뒤쪽에 붙인다."""
    if summary_df is None or len(summary_df.columns) == 0:
        return pd.DataFrame(columns=summary_columns)

    summary_df = summary_df.copy()

    # 예전 컬럼명 호환
    if "experiment_key" in summary_df.columns and "experiment_key" not in summary_df.columns:
        summary_df = summary_df.rename(columns={"experiment_key": "experiment_key"})
    if "created_dt" in summary_df.columns and "created_dt" not in summary_df.columns:
        summary_df = summary_df.rename(columns={"created_dt": "created_dt"})
    if "n_folds" in summary_df.columns and "n_folds" not in summary_df.columns:
        summary_df = summary_df.rename(columns={"n_folds": "n_folds"})

    for col in summary_columns:
        if col not in summary_df.columns:
            summary_df[col] = np.nan

    extra_cols = [c for c in summary_df.columns if c not in summary_columns]
    return summary_df[summary_columns + extra_cols]


def save_rolling_result_to_excel(result, config, experiment_dir=EXCEL_EXPERIMENT_DIR, run_name=None):
    """
    rolling 검증 결과를 experiment_summary.xlsx 하나에 누적 저장한다.

    저장 구조:
    experiments_excel/
      experiment_summary.xlsx    # summary 시트 하나만 사용
    """
    experiment_dir = Path(experiment_dir)
    experiment_dir.mkdir(parents=True, exist_ok=True)

    run_id = make_run_id(
        etf_code=config.get("etf_code"),
        model_name=config.get("model_name"),
        n_days=config.get("n_days"),
        threshold=config.get("threshold"),
        top_n=config.get("top_n"),
        run_name=run_name,
    )

    rolling_summary = result.get("rolling_summary", {})
    fold_metric_df = result.get("fold_metric_df", pd.DataFrame())
    errors = result.get("errors", [])

    experiment_key = make_experiment_key(config)

    summary_row = {
        "experiment_key": experiment_key,
        "created_dt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "run_id": run_id,
        **config,
        **rolling_summary,
        "n_folds_success": len(fold_metric_df),
        "error_count": len(errors),
    }

    summary_df = order_summary_df(make_one_row_df(summary_row))

    # 전체 summary 누적 파일 갱신. 개별 run_result.xlsx는 만들지 않는다.
    # master_summary_path = experiment_dir / "experiment_summary.xlsx"
    master_summary_path = experiment_dir / "experiment_summary2.xlsx"

    if master_summary_path.exists():
        try:
            old_summary_df = pd.read_excel(master_summary_path, sheet_name="summary")
            old_summary_df = order_summary_df(old_summary_df)
            master_summary_df = pd.concat([old_summary_df, summary_df], ignore_index=True)
        except Exception:
            master_summary_df = summary_df.copy()
    else:
        master_summary_df = summary_df.copy()

    # 혹시 같은 experiment_key가 중복으로 들어왔으면 최신 1개만 남긴다.
    if "experiment_key" in master_summary_df.columns:
        master_summary_df = master_summary_df.drop_duplicates(subset=["experiment_key"], keep="last")

    master_summary_df = order_summary_df(master_summary_df)

    sort_candidates = [
        "rolling_precision_mean",
        "rolling_recall_mean",
        "rolling_f1_mean",
        "rolling_auc_mean",
    ]
    sort_cols = [c for c in sort_candidates if c in master_summary_df.columns]

    if len(sort_cols) > 0:
        master_summary_df = master_summary_df.sort_values(sort_cols, ascending=False).reset_index(drop=True)

    write_excel_book(
        master_summary_path,
        {
            "summary": master_summary_df,
        }
    )

    return {
        "run_id": run_id,
        "experiment_key": experiment_key,
        "master_summary_path": str(master_summary_path),
    }

def run_rolling_validation(
    etf_code=ETF_CODE,
    n_days=N_DAYS,
    threshold=THRESHOLD,
    valid_months=VALID_MONTHS,
    vif_threshold=VIF_THRESHOLD,
    lag_search_years=LAG_SEARCH_YEARS,
    lag_days=LAG_DAYS,
    random_state=RANDOM_STATE,
    n_rf_runs=N_RF_RUNS,
    n_repeats=N_REPEATS,
    top_n=TOP_N,
    pred_threshold=PRED_THRESHOLD,
    model_name="random_forest",
    model_params=None,
    n_folds=10,
    step_months=1,
    start_date=START_DATE,
    end_date=END_DATE,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    base_df=None,
    base_feature_cols=None,
    close_col=None,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    run_name=None,
    verbose=True,
    fold_verbose=False
):
    """
    rolling 검증 실행.

    구조:
    - base_df를 한 번만 생성
    - fold 1: 최신 as_of_date 기준 최근 valid_months 검증
    - fold 2: as_of_date를 step_months만큼 과거로 이동
    - ... n_folds 반복
    - fold별 성능 평균/표준편차와 상세 결과를 Excel로 저장
    """

    if model_params is None:
        model_params = {}

    config = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "valid_months": valid_months,
        "vif_threshold": vif_threshold,
        "lag_search_years": lag_search_years,
        "lag_days": lag_days,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "pred_threshold": pred_threshold,
        "model_name": model_name,
        "model_params": model_params,
        "n_folds": n_folds,
        "step_months": step_months,
        "start_date": start_date,
        "end_date": end_date,
    }

    # 이미 같은 조건으로 저장된 실험이 있으면 재실행하지 않는다.
    if save_to_excel and skip_existing:
        existing_run = find_existing_experiment(config, experiment_dir=experiment_dir)
        if existing_run is not None:
            if verbose:
                print("\n기존에 같은 조건의 실험 결과가 있어 재실행을 건너뜁니다.")
                print("기존 run_id:", existing_run.get("run_id"))
                if "master_summary_path" in existing_run:
                    print("summary 파일:", existing_run.get("master_summary_path"))
            return {
                "skipped_existing": True,
                "existing_run": existing_run,
                "rolling_summary": {k: v for k, v in existing_run.items() if str(k).startswith("rolling_")},
                "fold_metric_df": pd.DataFrame(),
                "errors": [],
                "excel_save_info": {
                    "run_id": existing_run.get("run_id"),
                    "experiment_key": existing_run.get("experiment_key"),
                    "master_summary_path": str(Path(experiment_dir) / "experiment_summary.xlsx"),
                },
            }

    # =========================================================
    # 0. base_df 한 번만 생성
    # =========================================================
    if base_df is None:
        if verbose:
            print("=" * 80)
            print("Base dataset 생성 시작")
            print("=" * 80)

        base_df, base_feature_cols, close_col = make_base_feature_dataset(
            etf_code=etf_code,
            external_tickers=external_tickers,
            external_feature_types=external_feature_types,
            start_date=start_date,
            end_date=end_date
        )
    else:
        base_df = base_df.copy()
        if close_col is None:
            close_col = f"{etf_code}_adj_close"
        if base_feature_cols is None:
            base_feature_cols = [c for c in base_df.columns if c not in ["Date", close_col]]

    base_df["Date"] = pd.to_datetime(base_df["Date"])
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    as_of_dates = make_rolling_as_of_dates(
        base_df=base_df,
        n_folds=n_folds,
        step_months=step_months,
        end_date=end_date if end_date is not None else None,
        date_col="Date"
    )

    if verbose:
        print("=" * 80)
        print("Rolling 기준일")
        print("=" * 80)
        for i, d in enumerate(as_of_dates, start=1):
            print(f"fold {i:02d}: {d.date()}")

    fold_metric_list = []
    fold_feature_results = []
    fold_validation_results = []
    errors = []

    # =========================================================
    # 1. rolling fold 반복
    # =========================================================
    for fold, as_of_date in enumerate(as_of_dates, start=1):
        if verbose:
            print("\n" + "#" * 100)
            print(f"ROLLING FOLD {fold}/{len(as_of_dates)} | as_of_date={as_of_date.date()}")
            print("#" * 100)

        try:
            feature_result = build_top_feature_df(
                etf_code=etf_code,
                n_days=n_days,
                threshold=threshold,
                lag_search_years=lag_search_years,
                random_state=random_state,
                n_rf_runs=n_rf_runs,
                n_repeats=n_repeats,
                top_n=top_n,
                start_date=start_date,
                end_date=end_date,
                valid_months=valid_months,
                external_tickers=external_tickers,
                external_feature_types=external_feature_types,
                vif_threshold=vif_threshold,
                lag_days=lag_days,
                base_df=base_df,
                base_feature_cols=base_feature_cols,
                close_col=close_col,
                as_of_date=as_of_date,
                verbose=fold_verbose
            )

            valid_result = validate_top_feature_df(
                feature_result=feature_result,
                model_name=model_name,
                model_params=model_params,
                pred_threshold=pred_threshold,
                random_state=random_state,
                verbose=fold_verbose
            )

            metric_row = valid_result["metric_df"].iloc[0].to_dict()
            metric_row.update({
                "fold": fold,
                "as_of_date": as_of_date,
                "valid_start_date": feature_result.get("valid_start_date"),
                "valid_end_date": feature_result.get("as_of_date"),
                "error": None
            })

            fold_metric_list.append(pd.DataFrame([metric_row]))
            fold_feature_results.append({
                "fold": fold,
                "as_of_date": as_of_date,
                "feature_result": feature_result
            })
            fold_validation_results.append({
                "fold": fold,
                "as_of_date": as_of_date,
                "valid_result": valid_result
            })

            if verbose:
                print(
                    f"fold {fold:02d} 완료 | "
                    f"precision={metric_row.get('precision')} | "
                    f"pred_1_precision={metric_row.get('pred_1_precision')} | "
                    f"pred_1_count={metric_row.get('pred_1_count')} | "
                    f"eval_count={metric_row.get('eval_count')}"
                )

        except Exception as e:
            err = {
                "fold": fold,
                "as_of_date": as_of_date,
                "error": str(e)
            }
            errors.append(err)
            if verbose:
                print(f"[FOLD SKIP] fold {fold} 실패: {e}")

    if len(fold_metric_list) == 0:
        raise ValueError(f"성공한 rolling fold가 없습니다. errors={errors}")

    fold_metric_df = pd.concat(fold_metric_list, ignore_index=True)
    rolling_summary = summarize_rolling_metrics(fold_metric_df)

    all_selected_features_df, selected_features_summary_df = summarize_selected_features(fold_feature_results)

    # 보기 좋게 정렬
    sort_cols = ["rolling_precision_mean", "rolling_pred_1_precision_zero_fill_mean", "rolling_recall_mean", "rolling_f1_mean"]
    for c in sort_cols:
        if c not in rolling_summary:
            rolling_summary[c] = np.nan

    if verbose:
        print("\n" + "=" * 80)
        print("Rolling 검증 요약")
        print("=" * 80)
        print("성공 fold 수:", len(fold_metric_df))
        print("실패 fold 수:", len(errors))
        print("rolling_precision_mean:", rolling_summary.get("rolling_precision_mean"))
        print("rolling_pred_1_precision_zero_fill_mean:", rolling_summary.get("rolling_pred_1_precision_zero_fill_mean"))
        print("rolling_pred_1_count_mean:", rolling_summary.get("rolling_pred_1_count_mean"))
        display(fold_metric_df)
        display(selected_features_summary_df.head(30))

    result = {
        "fold_metric_df": fold_metric_df,
        "rolling_summary": rolling_summary,
        "all_selected_features_df": all_selected_features_df,
        "selected_features_summary_df": selected_features_summary_df,
        "fold_feature_results": fold_feature_results,
        "fold_validation_results": fold_validation_results,
        "errors": errors,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "close_col": close_col,
    }

    config = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "valid_months": valid_months,
        "vif_threshold": vif_threshold,
        "lag_search_years": lag_search_years,
        "lag_days": lag_days,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "pred_threshold": pred_threshold,
        "model_name": model_name,
        "model_params": model_params,
        "n_folds": n_folds,
        "step_months": step_months,
        "start_date": start_date,
        "end_date": end_date,
    }

    if save_to_excel:
        excel_save_info = save_rolling_result_to_excel(
            result=result,
            config=config,
            experiment_dir=experiment_dir,
            run_name=run_name,
        )
        result["excel_save_info"] = excel_save_info

        if verbose:
            print("\nExcel 저장 완료")
            print("전체 summary 파일:", excel_save_info["master_summary_path"])

    return result


def run_experiment_grid_rolling(
    experiment_grid,
    base_df=None,
    base_feature_cols=None,
    close_col=None,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    verbose=True,
    fold_verbose=False
):
    """
    여러 파라미터 조합을 rolling 검증으로 반복 실행한다.

    experiment_grid 예:
    [
        {
            "etf_code": "SMH",
            "n_days": 5,
            "threshold": 0.01,
            "model_name": "random_forest",
            "model_params": {"n_estimators": 500, "max_depth": None},
            "top_n": 30,
            "n_folds": 10,
        },
        ...
    ]
    """

    experiment_dir = Path(experiment_dir)
    results = []
    summary_rows = []

    # base_df를 넣지 않았으면 첫 번째 grid 기준으로 한 번만 생성
    if base_df is None:
        first = experiment_grid[0]
        etf_code = first.get("etf_code", ETF_CODE)
        start_date = first.get("start_date", START_DATE)
        end_date = first.get("end_date", END_DATE)
        external_tickers = first.get("external_tickers", EXTERNAL_TICKERS)
        external_feature_types = first.get("external_feature_types", EXTERNAL_FEATURE_TYPES)

        if verbose:
            print("=" * 80)
            print("Grid 공통 base_df 생성")
            print("=" * 80)

        base_df, base_feature_cols, close_col = make_base_feature_dataset(
            etf_code=etf_code,
            external_tickers=external_tickers,
            external_feature_types=external_feature_types,
            start_date=start_date,
            end_date=end_date
        )

    for i, cfg in enumerate(experiment_grid, start=1):
        if verbose:
            print("\n" + "=" * 100)
            print(f"EXPERIMENT {i}/{len(experiment_grid)}")
            print(cfg)
            print("=" * 100)

        result = run_rolling_validation(
            etf_code=cfg.get("etf_code", ETF_CODE),
            n_days=cfg.get("n_days", N_DAYS),
            threshold=cfg.get("threshold", THRESHOLD),
            valid_months=cfg.get("valid_months", VALID_MONTHS),
            vif_threshold=cfg.get("vif_threshold", VIF_THRESHOLD),
            lag_search_years=cfg.get("lag_search_years", LAG_SEARCH_YEARS),
            lag_days=cfg.get("lag_days", LAG_DAYS),
            random_state=cfg.get("random_state", RANDOM_STATE),
            n_rf_runs=cfg.get("n_rf_runs", N_RF_RUNS),
            n_repeats=cfg.get("n_repeats", N_REPEATS),
            top_n=cfg.get("top_n", TOP_N),
            pred_threshold=cfg.get("pred_threshold", PRED_THRESHOLD),
            model_name=cfg.get("model_name", "random_forest"),
            model_params=cfg.get("model_params", {}),
            n_folds=cfg.get("n_folds", 10),
            step_months=cfg.get("step_months", 1),
            start_date=cfg.get("start_date", START_DATE),
            end_date=cfg.get("end_date", END_DATE),
            external_tickers=cfg.get("external_tickers", EXTERNAL_TICKERS),
            external_feature_types=cfg.get("external_feature_types", EXTERNAL_FEATURE_TYPES),
            base_df=base_df,
            base_feature_cols=base_feature_cols,
            close_col=close_col,
            save_to_excel=save_to_excel,
            experiment_dir=experiment_dir,
            skip_existing=skip_existing,
            run_name=cfg.get("run_name"),
            verbose=verbose,
            fold_verbose=fold_verbose
        )

        results.append(result)

        row = cfg.copy()
        row.update(result["rolling_summary"])
        row["n_folds_success"] = len(result["fold_metric_df"])
        row["error_count"] = len(result["errors"])
        if "excel_save_info" in result:
            row.update(result["excel_save_info"])
        summary_rows.append(row)

    grid_summary_df = pd.DataFrame(summary_rows)

    if len(grid_summary_df) > 0:
        sort_candidates = [
            "rolling_precision_mean",
            "rolling_pred_1_precision_zero_fill_mean",
            "rolling_recall_mean",
            "rolling_f1_mean"
        ]
        sort_cols = [c for c in sort_candidates if c in grid_summary_df.columns]
        if len(sort_cols) > 0:
            grid_summary_df = grid_summary_df.sort_values(sort_cols, ascending=False).reset_index(drop=True)

    # 별도 grid_summary_*.xlsx는 만들지 않는다.
    # 각 실험 결과는 run_rolling_validation 내부에서 experiment_summary.xlsx 하나에 누적 저장된다.
    grid_excel_path = None
    if save_to_excel and verbose:
        print("\nGrid 결과는 experiment_summary.xlsx의 summary 시트에 누적 저장되었습니다.")

    if verbose:
        print("\n" + "=" * 80)
        print("GRID SUMMARY")
        print("=" * 80)
        display(grid_summary_df)

    return {
        "grid_summary_df": grid_summary_df,
        "results": results,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "close_col": close_col,
        "grid_excel_path": str(grid_excel_path) if grid_excel_path is not None else None,
    }


In [21]:

# # =========================================================
# # Rolling 검증 실행 예시: 단일 모델/단일 조합
# # - 결과는 experiments_excel 폴더의 Excel 파일로 저장됨
# # =========================================================

# rolling_result = run_rolling_validation(
#     etf_code=ETF_CODE,
#     n_days=N_DAYS,
#     threshold=THRESHOLD,
#     valid_months=VALID_MONTHS,
#     vif_threshold=VIF_THRESHOLD,
#     lag_search_years=LAG_SEARCH_YEARS,
#     lag_days=LAG_DAYS,
#     random_state=RANDOM_STATE,
#     n_rf_runs=N_RF_RUNS,
#     n_repeats=N_REPEATS,
#     top_n=TOP_N,
#     pred_threshold=PRED_THRESHOLD,
#     model_name="random_forest",
#     model_params={
#         "n_estimators": 500,
#         "max_depth": None,
#         "class_weight": "balanced"
#     },
#     n_folds=10,
#     step_months=1,
#     save_to_excel=True,
#     experiment_dir=EXCEL_EXPERIMENT_DIR,
#     verbose=True,
#     fold_verbose=False
# )

# fold_metric_df = rolling_result["fold_metric_df"]
# selected_features_summary_df = rolling_result["selected_features_summary_df"]
# rolling_summary = rolling_result["rolling_summary"]

# print(rolling_summary)


In [22]:

# # =========================================================
# # 여러 조합 Rolling 검증 예시
# # - 하나의 cfg가 Excel run 하나가 됨
# # - 처음에는 조합을 너무 많이 넣지 말고 2~4개 정도로 테스트 추천
# # =========================================================

# experiment_grid = [
#     {
#         "etf_code": "SMH",
#         "n_days": 5,
#         "threshold": 0.01,
#         "valid_months": 1,
#         "vif_threshold": 10,
#         "lag_search_years": 1,
#         "lag_days": [1, 3, 5, 10, 20, 40, 60, 120],
#         "top_n": 30,
#         "pred_threshold": 0.5,
#         "model_name": "random_forest",
#         "model_params": {"n_estimators": 500, "max_depth": None, "class_weight": "balanced"},
#         "n_folds": 10,
#         "step_months": 1,
#     },
#     {
#         "etf_code": "SMH",
#         "n_days": 5,
#         "threshold": 0.01,
#         "valid_months": 1,
#         "vif_threshold": 10,
#         "lag_search_years": 1,
#         "lag_days": [1, 3, 5, 10, 20, 40, 60, 120],
#         "top_n": 30,
#         "pred_threshold": 0.5,
#         "model_name": "extra_trees",
#         "model_params": {"n_estimators": 500, "max_depth": None, "class_weight": "balanced"},
#         "n_folds": 10,
#         "step_months": 1,
#     },
# ]

# grid_result = run_experiment_grid_rolling(
#     experiment_grid=experiment_grid,
#     save_to_excel=True,
#     experiment_dir=EXCEL_EXPERIMENT_DIR,
#     verbose=True,
#     fold_verbose=False
# )

# grid_summary_df = grid_result["grid_summary_df"]
# grid_summary_df


In [7]:

# =========================================================
# Optuna 자동 탐색
# - 사람이 직접 조합을 다 만들지 않고 Optuna가 조합을 선택
# - 결과 저장은 기존과 동일하게 experiments_excel/experiment_summary.xlsx 하나에 누적
# - 같은 조건은 experiment_key 기준으로 자동 스킵
# =========================================================

try:
    import optuna
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "optuna가 설치되어 있지 않습니다. 터미널에서 아래 명령 실행 후 커널을 재시작하세요.\n"
        "pip install optuna"
    ) from e


def suggest_model_params_for_optuna(trial, model_name):
    """Optuna trial에서 모델별 하이퍼파라미터를 제안한다."""
    model_name = str(model_name).lower()

    if model_name == "random_forest":
        return {
            "n_estimators": trial.suggest_categorical("rf_n_estimators", [300, 500, 800]),
            "max_depth": trial.suggest_categorical("rf_max_depth", [None, 3, 5, 7, 10]),
            "min_samples_leaf": trial.suggest_categorical("rf_min_samples_leaf", [1, 2, 5, 10]),
            "max_features": trial.suggest_categorical("rf_max_features", ["sqrt", "log2", 0.5]),
            "class_weight": "balanced",
        }

    if model_name == "extra_trees":
        return {
            "n_estimators": trial.suggest_categorical("et_n_estimators", [300, 500, 800]),
            "max_depth": trial.suggest_categorical("et_max_depth", [None, 3, 5, 7, 10]),
            "min_samples_leaf": trial.suggest_categorical("et_min_samples_leaf", [1, 2, 5, 10]),
            "max_features": trial.suggest_categorical("et_max_features", ["sqrt", "log2", 0.5]),
            "class_weight": "balanced",
        }

    if model_name == "gradient_boosting":
        return {
            "n_estimators": trial.suggest_categorical("gb_n_estimators", [100, 300, 500]),
            "learning_rate": trial.suggest_categorical("gb_learning_rate", [0.01, 0.03, 0.05, 0.1]),
            "max_depth": trial.suggest_categorical("gb_max_depth", [2, 3, 4, 5]),
        }

    if model_name == "hist_gradient_boosting":
        return {
            "max_iter": trial.suggest_categorical("hgb_max_iter", [100, 300, 500]),
            "learning_rate": trial.suggest_categorical("hgb_learning_rate", [0.01, 0.03, 0.05, 0.1]),
            "max_leaf_nodes": trial.suggest_categorical("hgb_max_leaf_nodes", [15, 31, 63]),
        }

    if model_name == "logistic":
        return {
            "C": trial.suggest_categorical("logistic_C", [0.01, 0.1, 1.0, 10.0]),
            "class_weight": "balanced",
            "max_iter": 3000,
        }

    return {}


def build_optuna_config(trial, fixed_cfg=None):
    """
    fixed_cfg는 고정값, trial은 탐색값을 만든다.
    fixed_cfg에 값이 있으면 그 값을 우선 사용하고, 없으면 Optuna가 탐색한다.
    """
    if fixed_cfg is None:
        fixed_cfg = {}

    cfg = dict(fixed_cfg)

    cfg.setdefault("etf_code", ETF_CODE)
    cfg.setdefault("valid_months", VALID_MONTHS)
    cfg.setdefault("vif_threshold", VIF_THRESHOLD)
    cfg.setdefault("lag_search_years", LAG_SEARCH_YEARS)
    cfg.setdefault("lag_days", LAG_DAYS)
    cfg.setdefault("random_state", RANDOM_STATE)
    cfg.setdefault("n_rf_runs", N_RF_RUNS)
    cfg.setdefault("n_repeats", N_REPEATS)
    cfg.setdefault("n_folds", 6)
    cfg.setdefault("step_months", 1)
    cfg.setdefault("start_date", START_DATE)
    cfg.setdefault("end_date", END_DATE)

    if "n_days" not in cfg:
        cfg["n_days"] = trial.suggest_categorical("n_days", [3, 5, 10])

    if "threshold" not in cfg:
        cfg["threshold"] = trial.suggest_categorical("threshold", [0.005, 0.01, 0.02, 0.03])

    if "top_n" not in cfg:
        cfg["top_n"] = trial.suggest_categorical("top_n", [10, 20, 30, 50])

    if "pred_threshold" not in cfg:
        cfg["pred_threshold"] = trial.suggest_float("pred_threshold", 0.25, 0.80, step=0.05)

    if "model_name" not in cfg:
        cfg["model_name"] = trial.suggest_categorical(
            "model_name",
            ["random_forest", "gradient_boosting", "hist_gradient_boosting", "logistic"]
        )

    if "model_params" not in cfg:
        cfg["model_params"] = suggest_model_params_for_optuna(trial, cfg["model_name"])

    # summary에서 Optuna 실험 여부를 구분하기 위한 보조 컬럼. experiment_key에는 포함되지 않는다.
    cfg["search_method"] = "optuna"
    cfg["optuna_trial_number"] = trial.number

    return cfg


def get_metric_from_result(result, metric_name, default=np.nan):
    """일반 실행 결과와 skip_existing 결과에서 metric을 안전하게 꺼낸다."""
    if result is None:
        return default

    if result.get("skipped_existing", False):
        existing_run = result.get("existing_run", {})
        return existing_run.get(metric_name, default)

    rolling_summary = result.get("rolling_summary", {})
    return rolling_summary.get(metric_name, default)


def get_n_folds_success_from_result(result):
    """일반 실행 결과와 skip_existing 결과에서 성공 fold 수를 안전하게 꺼낸다."""
    if result is None:
        return 0

    if result.get("skipped_existing", False):
        existing_run = result.get("existing_run", {})
        value = existing_run.get("n_folds_success", 0)
        try:
            return int(value)
        except Exception:
            return 0

    fold_metric_df = result.get("fold_metric_df", pd.DataFrame())
    return len(fold_metric_df)


def compute_optuna_score(
    result,
    n_folds,
    objective_metric="rolling_precision_mean",
    min_success_ratio=0.8,
    min_pred_1_count_mean=1.0,
):
    """
    Optuna가 최대화할 점수를 만든다.

    기본은 rolling_precision_mean을 최대화하되,
    - 성공 fold가 너무 적으면 감점
    - 예측 1이 거의 안 나오면 감점
    """
    base_score = get_metric_from_result(result, objective_metric, default=np.nan)
    pred_1_count_mean = get_metric_from_result(result, "rolling_pred_1_count_mean", default=np.nan)
    n_success = get_n_folds_success_from_result(result)

    if pd.isna(base_score):
        return 0.0

    success_ratio = n_success / max(int(n_folds), 1)
    score = float(base_score)

    # fold 성공률 패널티
    if success_ratio < min_success_ratio:
        score *= success_ratio / max(min_success_ratio, 1e-9)

    # 예측 1 개수가 너무 적으면 패널티
    if not pd.isna(pred_1_count_mean) and pred_1_count_mean < min_pred_1_count_mean:
        score *= max(float(pred_1_count_mean), 0.0) / max(float(min_pred_1_count_mean), 1e-9)

    return float(score)


def run_optuna_experiments(
    n_trials=30,
    fixed_cfg=None,
    objective_metric="rolling_precision_mean",
    min_success_ratio=0.8,
    min_pred_1_count_mean=1.0,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    verbose=True,
    fold_verbose=False,
):
    """
    Optuna로 실험 조건을 자동 탐색한다.

    Returns
    -------
    dict
        - study: Optuna study
        - trials_df: trial별 결과 요약
        - base_df/base_feature_cols/close_col: 재사용된 base dataset
    """
    if fixed_cfg is None:
        fixed_cfg = {}

    # Optuna 전체 trial에서 base_df는 한 번만 생성해서 재사용한다.
    etf_code = fixed_cfg.get("etf_code", ETF_CODE)
    start_date = fixed_cfg.get("start_date", START_DATE)
    end_date = fixed_cfg.get("end_date", END_DATE)
    external_tickers = fixed_cfg.get("external_tickers", EXTERNAL_TICKERS)
    external_feature_types = fixed_cfg.get("external_feature_types", EXTERNAL_FEATURE_TYPES)

    if verbose:
        print("=" * 80)
        print("Optuna 공통 base_df 생성")
        print("=" * 80)

    base_df, base_feature_cols, close_col = make_base_feature_dataset(
        etf_code=etf_code,
        external_tickers=external_tickers,
        external_feature_types=external_feature_types,
        start_date=start_date,
        end_date=end_date,
    )

    trial_rows = []

    def objective(trial):
        cfg = build_optuna_config(trial, fixed_cfg=fixed_cfg)

        if verbose:
            print("\n" + "=" * 100)
            print(f"OPTUNA TRIAL {trial.number + 1}/{n_trials}")
            print(cfg)
            print("=" * 100)

        try:
            result = run_rolling_validation(
                etf_code=cfg.get("etf_code", ETF_CODE),
                n_days=cfg.get("n_days", N_DAYS),
                threshold=cfg.get("threshold", THRESHOLD),
                valid_months=cfg.get("valid_months", VALID_MONTHS),
                vif_threshold=cfg.get("vif_threshold", VIF_THRESHOLD),
                lag_search_years=cfg.get("lag_search_years", LAG_SEARCH_YEARS),
                lag_days=cfg.get("lag_days", LAG_DAYS),
                random_state=cfg.get("random_state", RANDOM_STATE),
                n_rf_runs=cfg.get("n_rf_runs", N_RF_RUNS),
                n_repeats=cfg.get("n_repeats", N_REPEATS),
                top_n=cfg.get("top_n", TOP_N),
                pred_threshold=cfg.get("pred_threshold", PRED_THRESHOLD),
                model_name=cfg.get("model_name", "random_forest"),
                model_params=cfg.get("model_params", {}),
                n_folds=cfg.get("n_folds", 6),
                step_months=cfg.get("step_months", 1),
                start_date=cfg.get("start_date", START_DATE),
                end_date=cfg.get("end_date", END_DATE),
                external_tickers=external_tickers,
                external_feature_types=external_feature_types,
                base_df=base_df,
                base_feature_cols=base_feature_cols,
                close_col=close_col,
                save_to_excel=save_to_excel,
                experiment_dir=experiment_dir,
                skip_existing=skip_existing,
                run_name=f"optuna_trial_{trial.number:04d}",
                verbose=verbose,
                fold_verbose=fold_verbose,
            )

            score = compute_optuna_score(
                result=result,
                n_folds=cfg.get("n_folds", 6),
                objective_metric=objective_metric,
                min_success_ratio=min_success_ratio,
                min_pred_1_count_mean=min_pred_1_count_mean,
            )

            row = {
                "trial_number": trial.number,
                "score": score,
                **cfg,
                "n_folds_success": get_n_folds_success_from_result(result),
                objective_metric: get_metric_from_result(result, objective_metric),
                "rolling_pred_1_count_mean": get_metric_from_result(result, "rolling_pred_1_count_mean"),
                "skipped_existing": result.get("skipped_existing", False),
            }
            trial_rows.append(row)

            trial.set_user_attr("score", score)
            trial.set_user_attr("n_folds_success", row["n_folds_success"])
            trial.set_user_attr(objective_metric, row[objective_metric])
            trial.set_user_attr("rolling_pred_1_count_mean", row["rolling_pred_1_count_mean"])
            trial.set_user_attr("cfg", cfg)

            return score

        except Exception as e:
            if verbose:
                print(f"[OPTUNA TRIAL FAIL] trial={trial.number}, error={e}")

            trial_rows.append({
                "trial_number": trial.number,
                "score": 0.0,
                **cfg,
                "error": str(e),
            })
            return 0.0

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame(trial_rows)
    if len(trials_df) > 0 and "score" in trials_df.columns:
        trials_df = trials_df.sort_values("score", ascending=False).reset_index(drop=True)

    if verbose:
        print("\n" + "=" * 80)
        print("OPTUNA BEST TRIAL")
        print("=" * 80)
        print("best_value:", study.best_value)
        print("best_params:", study.best_params)
        display(trials_df.head(20))

    return {
        "study": study,
        "trials_df": trials_df,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "close_col": close_col,
    }


In [8]:
fixed_cfg = {
    "etf_code": "SMH",  
    # 예측 대상 ETF 코드
    # 예: SMH, QQQ, SPY, SOXX 등

    "valid_months": 3,  
    # fold 1개당 검증 구간 길이
    # 1이면 한 번 검증할 때 1개월 데이터를 valid로 사용

    "vif_threshold": 10,  
    # VIF 기반 다중공선성 제거 기준
    # 값이 낮을수록 비슷한 변수들을 더 강하게 제거
    # 보통 5~10 사용, 10은 무난한 기준

    "lag_search_years": 1,  
    # 변수별 최적 lag를 찾을 때 사용할 과거 기간
    # 1이면 각 fold 기준 최근 1년 데이터로 최적 lag 탐색

    "lag_days": [1, 3, 5, 10, 20, 40, 60, 120],  
    # 테스트할 lag 후보
    # 예: lag 5는 해당 변수가 5거래일 전에 움직인 값으로 타겟과 비교

    "random_state": 42,  
    # 랜덤 고정값
    # 같은 조건에서 결과가 최대한 동일하게 나오도록 고정

    "n_rf_runs": 3,  
    # feature importance 계산 시 RandomForest를 몇 번 반복할지
    # 여러 번 돌려 평균내면 변수 중요도가 조금 더 안정적임

    "n_repeats": 10,  
    # permutation importance 반복 횟수
    # 값이 클수록 중요도는 안정적이지만 실행 시간이 길어짐

    "n_folds": 10,  
    # rolling validation을 몇 번 수행할지
    # 10이면 최근 10개 fold를 검증

    "step_months": 1,  
    # fold를 몇 개월씩 이동할지
    # 1이면 valid 구간을 한 달씩 밀면서 검증
}
optuna_result = run_optuna_experiments(
    n_trials=30,
    fixed_cfg=fixed_cfg,
    objective_metric="rolling_precision_mean",
    min_success_ratio=0.8,
    min_pred_1_count_mean=1.0,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    verbose=True,
    fold_verbose=False,
)

optuna_trials_df = optuna_result["trials_df"]

Optuna 공통 base_df 생성
Loading external ticker: QQQ / QQQ
Loading external ticker: SPY / SPY
Loading external ticker: SOXX / SOXX
Loading external ticker: NVDA / NVDA
Loading external ticker: TSM / TSM
Loading external ticker: VIX / ^VIX
Loading external ticker: TNX / ^TNX
Loading external ticker: USDKRW / KRW=X
Loading external ticker: DXY / DX-Y.NYB
Loading external ticker: GOLD / GC=F
Loading external ticker: OIL / CL=F


[I 2026-05-23 19:08:03,008] A new study created in memory with name: no-name-d8c09092-3692-45d3-ae4c-c5b7586fe239



OPTUNA TRIAL 1/30
{'etf_code': 'SMH', 'valid_months': 3, 'vif_threshold': 10, 'lag_search_years': 1, 'lag_days': [1, 3, 5, 10, 20, 40, 60, 120], 'random_state': 42, 'n_rf_runs': 3, 'n_repeats': 10, 'n_folds': 10, 'step_months': 1, 'start_date': '2020-01-01', 'end_date': None, 'n_days': 10, 'threshold': 0.02, 'top_n': 50, 'pred_threshold': 0.65, 'model_name': 'gradient_boosting', 'model_params': {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2}, 'search_method': 'optuna', 'optuna_trial_number': 0}
Rolling 기준일
fold 01: 2026-05-22
fold 02: 2026-04-22
fold 03: 2026-03-20
fold 04: 2026-02-20
fold 05: 2026-01-22
fold 06: 2025-12-22
fold 07: 2025-11-21
fold 08: 2025-10-22
fold 09: 2025-09-22
fold 10: 2025-08-22

####################################################################################################
ROLLING FOLD 1/10 | as_of_date=2026-05-22
####################################################################################################
VIF 계산 대상 row 수: 1484
VIF 계산 

,etf_code,n_days,threshold,lag_search_years,top_n,feature_count,model_name,model_params,eval_count,pred_1_count,...,logloss,tn,fp,fn,tp,fold,as_of_date,valid_start_date,valid_end_date,error
0,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",54,0,...,0.741547,20,0,34,0,1,2026-05-22,2026-02-22,2026-05-22,None
1,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",53,4,...,0.684085,31,3,18,1,2,2026-04-22,2026-01-22,2026-04-22,None
2,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",51,0,...,0.704068,29,0,22,0,3,2026-03-20,2025-12-20,2026-03-20,None
3,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",52,3,...,0.712177,19,3,30,0,4,2026-02-20,2025-11-20,2026-02-20,None
4,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",53,6,...,0.724722,22,3,25,3,5,2026-01-22,2025-10-22,2026-01-22,None
5,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",55,11,...,0.740325,22,6,22,5,6,2025-12-22,2025-09-22,2025-12-22,None
6,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",56,7,...,0.822918,16,6,33,1,7,2025-11-21,2025-08-21,2025-11-21,None
7,SMH,10,0.02,1,50,27,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",56,4,...,0.723811,24,2,28,2,8,2025-10-22,2025-07-22,2025-10-22,None
8,SMH,10,0.02,1,50,26,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",54,2,...,0.777236,27,2,25,0,9,2025-09-22,2025-06-22,2025-09-22,None
9,SMH,10,0.02,1,50,26,gradient_boosting,"{'n_estimators': 100, 'learning_rate': 0.05, '...",54,2,...,0.798978,15,2,37,0,10,2025-08-22,2025-05-22,2025-08-22,None


,feature,selected_count,avg_importance_score,avg_importance_mean,avg_importance_std,avg_selected_lag,first_base_feature
0,TSM_ret_20d_lag40,10,0.051973,0.053624,0.001651,40.0,TSM_ret_20d
1,TNX_level_lag120,8,0.086924,0.089328,0.002404,120.0,TNX_level
2,OIL_ret_20d_lag40,8,0.070253,0.072742,0.002489,40.0,OIL_ret_20d
3,USDKRW_ret_20d_lag120,8,0.058933,0.060822,0.001888,120.0,USDKRW_ret_20d
4,VIX_level_lag40,8,0.056319,0.058300,0.001981,40.0,VIX_level
5,DXY_ret_20d_lag120,7,0.056765,0.058590,0.001825,120.0,DXY_ret_20d
6,VIX_chg_20d_lag120,7,0.046516,0.047975,0.001459,120.0,VIX_chg_20d
7,USDKRW_ret_5d_lag120,7,0.038063,0.039531,0.001468,120.0,USDKRW_ret_5d
8,NVDA_ret_5d_lag60,7,0.034626,0.036212,0.001586,60.0,NVDA_ret_5d
9,SPY_ret_5d_lag60,7,0.029618,0.030801,0.001183,60.0,SPY_ret_5d


[I 2026-05-23 19:10:51,991] Trial 0 finished with value: 0.18474025974025973 and parameters: {'n_days': 10, 'threshold': 0.02, 'top_n': 50, 'pred_threshold': 0.65, 'model_name': 'gradient_boosting', 'gb_n_estimators': 100, 'gb_learning_rate': 0.05, 'gb_max_depth': 2}. Best is trial 0 with value: 0.18474025974025973.



Excel 저장 완료
전체 summary 파일: experiments_excel/experiment_summary2.xlsx

OPTUNA TRIAL 2/30
{'etf_code': 'SMH', 'valid_months': 3, 'vif_threshold': 10, 'lag_search_years': 1, 'lag_days': [1, 3, 5, 10, 20, 40, 60, 120], 'random_state': 42, 'n_rf_runs': 3, 'n_repeats': 10, 'n_folds': 10, 'step_months': 1, 'start_date': '2020-01-01', 'end_date': None, 'n_days': 5, 'threshold': 0.005, 'top_n': 50, 'pred_threshold': 0.35, 'model_name': 'logistic', 'model_params': {'C': 1.0, 'class_weight': 'balanced', 'max_iter': 3000}, 'search_method': 'optuna', 'optuna_trial_number': 1}
Rolling 기준일
fold 01: 2026-05-22
fold 02: 2026-04-22
fold 03: 2026-03-20
fold 04: 2026-02-20
fold 05: 2026-01-22
fold 06: 2025-12-22
fold 07: 2025-11-21
fold 08: 2025-10-22
fold 09: 2025-09-22
fold 10: 2025-08-22

####################################################################################################
ROLLING FOLD 1/10 | as_of_date=2026-05-22
########################################################################

[W 2026-05-23 19:11:40,005] Trial 1 failed with parameters: {'n_days': 5, 'threshold': 0.005, 'top_n': 50, 'pred_threshold': 0.35, 'model_name': 'logistic', 'logistic_C': 1.0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/jongheelee/Desktop/JH/01. Working/1. ETF_Prediction/etf_predict/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/8q/_1f4z4tj6zqgt3d_xgf1mz280000gn/T/ipykernel_77754/2807428258.py", line 236, in objective
    result = run_rolling_validation(
             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/8q/_1f4z4tj6zqgt3d_xgf1mz280000gn/T/ipykernel_77754/1527653700.py", line 955, in run_rolling_validation
    feature_result = build_top_feature_df(
                     ^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/8q/_1f4z4tj6zqgt3d_xgf1mz280000gn/T/ipykernel_77754/1527653700.py", line 251, in bui

KeyboardInterrupt: 

## Holdout 방식 Optuna 추가

마지막 `test_months`개월을 test로 분리하고, 그 이전 train 데이터로 변수 선택/학습 후 test를 예측합니다. 선택된 변수 리스트도 summary에 저장됩니다.

In [9]:

# =========================================================
# Holdout 방식 Optuna 실험
# - fold/rolling 검증 제거
# - 마지막 test_months 개월을 test로 분리
# - test 시작 전까지의 train 데이터로 feature selection 1회
# - 같은 selected feature로 train 학습 후 test 예측
# - 선택된 변수 리스트까지 experiment_summary_holdout.xlsx에 저장
# =========================================================

HOLDOUT_SUMMARY_FILE_NAME = "experiment_summary_holdout.xlsx"

HOLDOUT_EXPERIMENT_KEY_COLS = [
    "etf_code",
    "n_days",
    "threshold",
    "test_months",
    "vif_threshold",
    "lag_search_years",
    "lag_days",
    "random_state",
    "n_rf_runs",
    "n_repeats",
    "top_n",
    "pred_threshold",
    "model_name",
    "model_params",
    "start_date",
    "end_date",
]

HOLDOUT_SUMMARY_COLUMNS = [
    "experiment_key",
    "created_dt",
    "run_id",
    "etf_code",
    "test_precision",
    "n_days",
    "threshold",
    "test_months",
    "vif_threshold",
    "lag_search_years",
    "lag_days",
    "top_n",
    "selected_feature_count",
    "selected_feature_list",
    "selected_feature_top10",
    "pred_threshold",
    "model_name",
    "model_params",
    "test_accuracy",
    "test_recall",
    "test_f1",
    "test_auc",
    "test_logloss",
    "test_eval_count",
    "test_pred_1_count",
    "test_actual_1_count",
    "test_pred_1_actual_1_count",
    "test_pred_1_precision",
    "test_tn",
    "test_fp",
    "test_fn",
    "test_tp",
    "error_count",
]


def make_holdout_experiment_key(config):
    """holdout 방식 동일 실험 여부 판단 key."""
    key_dict = {
        col: normalize_experiment_value(config.get(col))
        for col in HOLDOUT_EXPERIMENT_KEY_COLS
    }
    key_json = json.dumps(key_dict, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.md5(key_json.encode("utf-8")).hexdigest()


def load_holdout_summary(experiment_dir=EXCEL_EXPERIMENT_DIR):
    """holdout summary Excel을 읽는다. 없으면 빈 DataFrame 반환."""
    path = Path(experiment_dir) / HOLDOUT_SUMMARY_FILE_NAME
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(path, sheet_name="summary")
    except Exception:
        return pd.DataFrame()


def find_existing_holdout_experiment(config, experiment_dir=EXCEL_EXPERIMENT_DIR):
    """같은 holdout 실험 조건이 이미 저장되어 있는지 찾는다."""
    summary_df = load_holdout_summary(experiment_dir)
    if summary_df.empty:
        return None

    target_key = make_holdout_experiment_key(config)
    if "experiment_key" in summary_df.columns:
        matched = summary_df[summary_df["experiment_key"].astype(str) == str(target_key)]
        if len(matched) > 0:
            return matched.iloc[0].to_dict()
    return None


def order_holdout_summary_df(summary_df, summary_columns=HOLDOUT_SUMMARY_COLUMNS):
    """holdout summary 컬럼 순서를 고정한다."""
    if summary_df is None or len(summary_df.columns) == 0:
        return pd.DataFrame(columns=summary_columns)

    summary_df = summary_df.copy()
    for col in summary_columns:
        if col not in summary_df.columns:
            summary_df[col] = np.nan

    extra_cols = [c for c in summary_df.columns if c not in summary_columns]
    return summary_df[summary_columns + extra_cols]


def make_selected_feature_summary_values(top_feature_df, max_chars=30000):
    """top_feature_df에서 Excel summary용 선택 변수 요약값을 만든다."""
    if top_feature_df is None or len(top_feature_df) == 0:
        return {
            "selected_feature_count": 0,
            "selected_feature_list": "",
            "selected_feature_top10": "",
        }

    df = top_feature_df.copy()
    if "feature" not in df.columns:
        return {
            "selected_feature_count": 0,
            "selected_feature_list": "",
            "selected_feature_top10": "",
        }

    feature_list = df["feature"].astype(str).tolist()
    selected_feature_list = " | ".join(feature_list)

    top10_items = []
    for _, row in df.head(10).iterrows():
        feature = str(row.get("feature", ""))
        score = row.get("importance_score", np.nan)
        if pd.notna(score):
            top10_items.append(f"{feature}({float(score):.6f})")
        else:
            top10_items.append(feature)

    selected_feature_top10 = " | ".join(top10_items)

    # Excel 셀 문자 수 제한 방어
    if len(selected_feature_list) > max_chars:
        selected_feature_list = selected_feature_list[:max_chars] + " ...[truncated]"

    return {
        "selected_feature_count": len(feature_list),
        "selected_feature_list": selected_feature_list,
        "selected_feature_top10": selected_feature_top10,
    }


def build_top_feature_df_from_base(
    base_df,
    base_feature_cols,
    close_col,
    etf_code,
    n_days,
    threshold,
    test_months=3,
    lag_search_years=1,
    random_state=42,
    n_rf_runs=3,
    n_repeats=10,
    top_n=30,
    start_date=START_DATE,
    end_date=END_DATE,
    vif_threshold=VIF_THRESHOLD,
    lag_days=LAG_DAYS,
    verbose=True,
):
    """
    holdout 방식 feature selection.

    핵심 구조:
    - base_df의 마지막 test_months 개월은 test로 분리
    - test 시작일 전까지의 train 데이터만 사용해 VIF / target / lag / importance / top_n 선택
    - 선택된 변수와 lag는 이후 test 검증에서 고정 사용
    """
    base_df = base_df.copy().sort_values("Date").reset_index(drop=True)

    max_date = base_df["Date"].max()
    test_start_date = max_date - pd.DateOffset(months=test_months)

    train_base_df = base_df[base_df["Date"] < test_start_date].copy()
    valid_base_df = base_df[base_df["Date"] >= test_start_date].copy()

    if verbose:
        print("=" * 60)
        print("Holdout feature selection")
        print("train 기간:", train_base_df["Date"].min(), "~", train_base_df["Date"].max())
        print("test  기간:", valid_base_df["Date"].min(), "~", valid_base_df["Date"].max())
        print("train_base_df shape:", train_base_df.shape)
        print("test_base_df shape:", valid_base_df.shape)
        print("close_col:", close_col)

    # 1. VIF는 train 데이터에서만 수행
    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=base_feature_cols,
        vif_threshold=vif_threshold,
        date_col="Date",
        verbose=verbose,
    )

    keep_cols = ["Date", close_col] + vif_feature_cols
    vif_filtered_df = train_base_df[keep_cols].copy()

    # 2. target도 train 데이터에서만 생성
    target_df, target_col = add_target_column(
        df=vif_filtered_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold,
    )
    target_df = target_df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    if target_df[target_col].nunique() < 2:
        raise ValueError(f"train target class가 1개뿐입니다. target 분포: {target_df[target_col].value_counts().to_dict()}")

    if verbose:
        print("target_col:", target_col)
        print("target_df shape:", target_df.shape)
        print("target 분포:")
        print(target_df[target_col].value_counts())

    # 3. lag 탐색도 train 데이터의 최근 lag_search_years 구간에서만 수행
    max_train_target_date = target_df["Date"].max()
    lag_search_start_date = max_train_target_date - pd.DateOffset(years=lag_search_years)
    target_df_for_lag_search = target_df[target_df["Date"] >= lag_search_start_date].copy()

    exclude_cols_for_lag = [
        "Date",
        close_col,
        f"future_ret_{n_days}d",
        target_col,
    ]
    lag_search_feature_cols = [
        col for col in target_df.columns
        if col not in exclude_cols_for_lag
    ]

    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=target_df_for_lag_search,
        feature_cols=lag_search_feature_cols,
        target_col=target_col,
        lag_days=lag_days,
        date_col="Date",
    )

    # 4. train 데이터에 best lag 적용
    lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date",
    )

    if lagged_df[target_col].nunique() < 2:
        raise ValueError(f"lagged train target class가 1개뿐입니다. target 분포: {lagged_df[target_col].value_counts().to_dict()}")

    if verbose:
        print("lagged_df shape:", lagged_df.shape)
        print("lagged feature count:", len(lagged_feature_cols))

    # 5. train 데이터에서 RF permutation importance로 top feature 선택
    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=n_rf_runs,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    importance_view_df = importance_df.copy()
    importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    importance_with_lag_df = importance_view_df.merge(
        best_lag_df.rename(columns={
            "feature": "base_feature",
            "lag": "best_lag",
            "corr": "lag_corr",
            "abs_corr": "lag_abs_corr",
            "n_rows": "lag_n_rows",
        }),
        on="base_feature",
        how="left",
    )

    importance_with_lag_df = importance_with_lag_df.sort_values(
        "importance_score",
        ascending=False,
    ).reset_index(drop=True)

    top_feature_df = importance_with_lag_df.head(top_n).copy()
    top_feature_cols = top_feature_df["feature"].tolist()

    if verbose:
        print("TOP feature count:", len(top_feature_cols))
        print(top_feature_cols)
        display(top_feature_df)

    return {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "test_months": test_months,
        "lag_search_years": lag_search_years,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "start_date": start_date,
        "end_date": end_date,
        "vif_threshold": vif_threshold,
        "lag_days": lag_days,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "train_base_df": train_base_df,
        "valid_base_df": valid_base_df,  # validate_top_feature_df 호환을 위해 이름 유지. 실제 의미는 test_base_df.
        "test_base_df": valid_base_df,
        "test_start_date": test_start_date,
        "close_col": close_col,
        "vif_feature_cols": vif_feature_cols,
        "removed_vif_df": removed_vif_df,
        "final_vif_df": final_vif_df,
        "vif_filtered_df": vif_filtered_df,
        "target_df": target_df,
        "target_col": target_col,
        "lag_result_df": lag_result_df,
        "best_lag_df": best_lag_df,
        "lagged_df": lagged_df,
        "lagged_feature_cols": lagged_feature_cols,
        "importance_df": importance_df,
        "raw_importance_df": raw_importance_df,
        "baseline_df": baseline_df,
        "importance_with_lag_df": importance_with_lag_df,
        "top_feature_df": top_feature_df,
        "top_feature_cols": top_feature_cols,
    }


def summarize_holdout_metrics(metric_df):
    """validate_top_feature_df의 metric_df를 test_* 컬럼으로 변환한다."""
    if metric_df is None or len(metric_df) == 0:
        return {
            "test_accuracy": np.nan,
            "test_precision": np.nan,
            "test_recall": np.nan,
            "test_f1": np.nan,
            "test_auc": np.nan,
            "test_logloss": np.nan,
            "test_eval_count": 0,
            "test_pred_1_count": 0,
            "test_actual_1_count": np.nan,
            "test_pred_1_actual_1_count": np.nan,
            "test_pred_1_precision": np.nan,
            "test_tn": np.nan,
            "test_fp": np.nan,
            "test_fn": np.nan,
            "test_tp": np.nan,
        }

    row = metric_df.iloc[0].to_dict()
    return {
        "test_accuracy": row.get("accuracy", np.nan),
        "test_precision": row.get("precision", np.nan),
        "test_recall": row.get("recall", np.nan),
        "test_f1": row.get("f1", np.nan),
        "test_auc": row.get("auc", np.nan),
        "test_logloss": row.get("logloss", np.nan),
        "test_eval_count": row.get("eval_count", 0),
        "test_pred_1_count": row.get("pred_1_count", 0),
        "test_actual_1_count": row.get("actual_1_count", np.nan),
        "test_pred_1_actual_1_count": row.get("pred_1_actual_1_count", np.nan),
        "test_pred_1_precision": row.get("pred_1_precision", np.nan),
        "test_tn": row.get("tn", np.nan),
        "test_fp": row.get("fp", np.nan),
        "test_fn": row.get("fn", np.nan),
        "test_tp": row.get("tp", np.nan),
    }


def save_holdout_result_to_excel(result, config, experiment_dir=EXCEL_EXPERIMENT_DIR, run_name=None):
    """holdout 검증 결과를 experiment_summary_holdout.xlsx 하나에 누적 저장한다."""
    experiment_dir = Path(experiment_dir)
    experiment_dir.mkdir(parents=True, exist_ok=True)

    run_id = make_run_id(
        etf_code=config.get("etf_code"),
        model_name=config.get("model_name"),
        n_days=config.get("n_days"),
        threshold=config.get("threshold"),
        top_n=config.get("top_n"),
        run_name=run_name,
    )

    experiment_key = make_holdout_experiment_key(config)
    errors = result.get("errors", [])

    metric_df = result.get("metric_df", pd.DataFrame())
    test_summary = summarize_holdout_metrics(metric_df)

    feature_result = result.get("feature_result", {})
    selected_summary = make_selected_feature_summary_values(
        feature_result.get("top_feature_df", pd.DataFrame())
    )

    summary_row = {
        "experiment_key": experiment_key,
        "created_dt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "run_id": run_id,
        **config,
        **selected_summary,
        **test_summary,
        "error_count": len(errors),
    }

    summary_df = order_holdout_summary_df(make_one_row_df(summary_row))
    master_summary_path = experiment_dir / HOLDOUT_SUMMARY_FILE_NAME

    if master_summary_path.exists():
        try:
            old_summary_df = pd.read_excel(master_summary_path, sheet_name="summary")
            old_summary_df = order_holdout_summary_df(old_summary_df)
            master_summary_df = pd.concat([old_summary_df, summary_df], ignore_index=True)
        except Exception:
            master_summary_df = summary_df.copy()
    else:
        master_summary_df = summary_df.copy()

    if "experiment_key" in master_summary_df.columns:
        master_summary_df = master_summary_df.drop_duplicates(subset=["experiment_key"], keep="last")

    master_summary_df = order_holdout_summary_df(master_summary_df)

    sort_cols = [c for c in ["test_precision", "test_f1", "test_recall", "test_auc"] if c in master_summary_df.columns]
    if len(sort_cols) > 0:
        master_summary_df = master_summary_df.sort_values(sort_cols, ascending=False).reset_index(drop=True)

    write_excel_book(master_summary_path, {"summary": master_summary_df})

    return {
        "run_id": run_id,
        "experiment_key": experiment_key,
        "master_summary_path": str(master_summary_path),
    }


def run_holdout_validation(
    etf_code=ETF_CODE,
    n_days=N_DAYS,
    threshold=THRESHOLD,
    test_months=3,
    vif_threshold=VIF_THRESHOLD,
    lag_search_years=LAG_SEARCH_YEARS,
    lag_days=LAG_DAYS,
    random_state=RANDOM_STATE,
    n_rf_runs=N_RF_RUNS,
    n_repeats=N_REPEATS,
    top_n=TOP_N,
    pred_threshold=PRED_THRESHOLD,
    model_name="random_forest",
    model_params=None,
    start_date=START_DATE,
    end_date=END_DATE,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    base_df=None,
    base_feature_cols=None,
    close_col=None,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    run_name=None,
    verbose=True,
):
    """
    holdout 검증 실행.

    구조:
    1. 전체 데이터의 마지막 test_months 개월을 test로 분리
    2. test 시작 전까지의 train 데이터로 feature selection 1회
    3. 같은 selected feature로 train 학습
    4. 마지막 test_months 개월 예측 검증
    """
    if model_params is None:
        model_params = {}

    config = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "test_months": test_months,
        "vif_threshold": vif_threshold,
        "lag_search_years": lag_search_years,
        "lag_days": lag_days,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "pred_threshold": pred_threshold,
        "model_name": model_name,
        "model_params": model_params,
        "start_date": start_date,
        "end_date": end_date,
    }

    if save_to_excel and skip_existing:
        existing_run = find_existing_holdout_experiment(config, experiment_dir=experiment_dir)
        if existing_run is not None:
            if verbose:
                print("\n기존에 같은 조건의 holdout 실험 결과가 있어 재실행을 건너뜁니다.")
                print("기존 run_id:", existing_run.get("run_id"))
                print("기존 test_precision:", existing_run.get("test_precision"))
            return {
                "skipped_existing": True,
                "existing_run": existing_run,
                "metric_df": pd.DataFrame(),
                "errors": [],
                "excel_save_info": {
                    "run_id": existing_run.get("run_id"),
                    "experiment_key": existing_run.get("experiment_key"),
                    "master_summary_path": str(Path(experiment_dir) / HOLDOUT_SUMMARY_FILE_NAME),
                },
            }

    if base_df is None or base_feature_cols is None or close_col is None:
        if verbose:
            print("=" * 80)
            print("base_df 생성")
            print("=" * 80)
        base_df, base_feature_cols, close_col = make_base_feature_dataset(
            etf_code=etf_code,
            external_tickers=external_tickers,
            external_feature_types=external_feature_types,
            start_date=start_date,
            end_date=end_date,
        )

    errors = []

    try:
        feature_result = build_top_feature_df_from_base(
            base_df=base_df,
            base_feature_cols=base_feature_cols,
            close_col=close_col,
            etf_code=etf_code,
            n_days=n_days,
            threshold=threshold,
            test_months=test_months,
            lag_search_years=lag_search_years,
            random_state=random_state,
            n_rf_runs=n_rf_runs,
            n_repeats=n_repeats,
            top_n=top_n,
            start_date=start_date,
            end_date=end_date,
            vif_threshold=vif_threshold,
            lag_days=lag_days,
            verbose=verbose,
        )

        valid_result = validate_top_feature_df(
            feature_result=feature_result,
            model_name=model_name,
            model_params=model_params,
            pred_threshold=pred_threshold,
            random_state=random_state,
            n_days=n_days,
            threshold=threshold,
            verbose=verbose,
        )

        result = {
            "config": config,
            "feature_result": feature_result,
            "valid_result": valid_result,
            "metric_df": valid_result.get("metric_df", pd.DataFrame()),
            "valid_pred_df": valid_result.get("valid_pred_df", pd.DataFrame()),
            "eval_df": valid_result.get("eval_df", pd.DataFrame()),
            "errors": errors,
        }

    except Exception as e:
        errors.append({"error": str(e)})
        if verbose:
            print("[HOLDOUT FAIL]", e)
        result = {
            "config": config,
            "feature_result": {},
            "valid_result": {},
            "metric_df": pd.DataFrame(),
            "valid_pred_df": pd.DataFrame(),
            "eval_df": pd.DataFrame(),
            "errors": errors,
        }

    if save_to_excel:
        excel_save_info = save_holdout_result_to_excel(
            result=result,
            config=config,
            experiment_dir=experiment_dir,
            run_name=run_name,
        )
        result["excel_save_info"] = excel_save_info
        if verbose:
            print("\nExcel 저장 완료")
            print("holdout summary 파일:", excel_save_info["master_summary_path"])

    return result


def build_optuna_holdout_config(trial, fixed_cfg=None):
    """holdout 방식 Optuna config 생성."""
    if fixed_cfg is None:
        fixed_cfg = {}

    cfg = dict(fixed_cfg)

    cfg.setdefault("etf_code", ETF_CODE)
    cfg.setdefault("test_months", 3)
    cfg.setdefault("vif_threshold", VIF_THRESHOLD)
    cfg.setdefault("lag_search_years", LAG_SEARCH_YEARS)
    cfg.setdefault("lag_days", LAG_DAYS)
    cfg.setdefault("random_state", RANDOM_STATE)
    cfg.setdefault("n_rf_runs", N_RF_RUNS)
    cfg.setdefault("n_repeats", N_REPEATS)
    cfg.setdefault("start_date", START_DATE)
    cfg.setdefault("end_date", END_DATE)

    if "n_days" not in cfg:
        cfg["n_days"] = trial.suggest_categorical("n_days", [3, 5, 10, 20])

    if "threshold" not in cfg:
        cfg["threshold"] = trial.suggest_categorical("threshold", [0.005, 0.01, 0.015, 0.02, 0.03])

    if "top_n" not in cfg:
        cfg["top_n"] = trial.suggest_categorical("top_n", [10, 20, 30, 50])

    if "pred_threshold" not in cfg:
        cfg["pred_threshold"] = trial.suggest_float("pred_threshold", 0.25, 0.80, step=0.05)

    if "model_name" not in cfg:
        cfg["model_name"] = trial.suggest_categorical(
            "model_name",
            ["random_forest", "gradient_boosting", "hist_gradient_boosting", "logistic"],
        )

    if "model_params" not in cfg:
        cfg["model_params"] = suggest_model_params_for_optuna(trial, cfg["model_name"])

    cfg["search_method"] = "optuna_holdout"
    cfg["optuna_trial_number"] = trial.number

    return cfg


def get_holdout_metric_from_result(result, metric_name, default=np.nan):
    """holdout result 또는 skipped_existing에서 metric을 안전하게 가져온다."""
    if result is None:
        return default

    if result.get("skipped_existing", False):
        return result.get("existing_run", {}).get(metric_name, default)

    metric_df = result.get("metric_df", pd.DataFrame())
    test_summary = summarize_holdout_metrics(metric_df)
    return test_summary.get(metric_name, default)


def compute_optuna_holdout_score(
    result,
    objective_metric="test_precision",
    min_eval_count=10,
    min_pred_1_count=3,
):
    """
    holdout Optuna 점수.
    기본은 test_precision 최대화.
    단, 평가 row나 예측 1 개수가 너무 적으면 감점한다.
    """
    base_score = get_holdout_metric_from_result(result, objective_metric, default=np.nan)
    eval_count = get_holdout_metric_from_result(result, "test_eval_count", default=0)
    pred_1_count = get_holdout_metric_from_result(result, "test_pred_1_count", default=0)

    if pd.isna(base_score):
        return 0.0

    score = float(base_score)
    score *= min(1.0, float(eval_count) / max(float(min_eval_count), 1e-9))
    score *= min(1.0, float(pred_1_count) / max(float(min_pred_1_count), 1e-9))
    return float(score)


def run_optuna_holdout_experiments(
    n_trials=30,
    fixed_cfg=None,
    objective_metric="test_precision",
    min_eval_count=10,
    min_pred_1_count=3,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    verbose=True,
):
    """
    Optuna로 holdout 실험 조건을 자동 탐색한다.

    결과는 experiment_summary_holdout.xlsx의 summary 시트에 누적된다.
    """
    if fixed_cfg is None:
        fixed_cfg = {}

    etf_code = fixed_cfg.get("etf_code", ETF_CODE)
    start_date = fixed_cfg.get("start_date", START_DATE)
    end_date = fixed_cfg.get("end_date", END_DATE)
    external_tickers = fixed_cfg.get("external_tickers", EXTERNAL_TICKERS)
    external_feature_types = fixed_cfg.get("external_feature_types", EXTERNAL_FEATURE_TYPES)

    if verbose:
        print("=" * 80)
        print("Optuna holdout 공통 base_df 생성")
        print("=" * 80)

    base_df, base_feature_cols, close_col = make_base_feature_dataset(
        etf_code=etf_code,
        external_tickers=external_tickers,
        external_feature_types=external_feature_types,
        start_date=start_date,
        end_date=end_date,
    )

    trial_rows = []

    def objective(trial):
        cfg = build_optuna_holdout_config(trial, fixed_cfg=fixed_cfg)

        if verbose:
            print("\n" + "=" * 100)
            print(f"OPTUNA HOLDOUT TRIAL {trial.number + 1}/{n_trials}")
            print(cfg)
            print("=" * 100)

        try:
            result = run_holdout_validation(
                etf_code=cfg.get("etf_code", ETF_CODE),
                n_days=cfg.get("n_days", N_DAYS),
                threshold=cfg.get("threshold", THRESHOLD),
                test_months=cfg.get("test_months", 3),
                vif_threshold=cfg.get("vif_threshold", VIF_THRESHOLD),
                lag_search_years=cfg.get("lag_search_years", LAG_SEARCH_YEARS),
                lag_days=cfg.get("lag_days", LAG_DAYS),
                random_state=cfg.get("random_state", RANDOM_STATE),
                n_rf_runs=cfg.get("n_rf_runs", N_RF_RUNS),
                n_repeats=cfg.get("n_repeats", N_REPEATS),
                top_n=cfg.get("top_n", TOP_N),
                pred_threshold=cfg.get("pred_threshold", PRED_THRESHOLD),
                model_name=cfg.get("model_name", "random_forest"),
                model_params=cfg.get("model_params", {}),
                start_date=cfg.get("start_date", START_DATE),
                end_date=cfg.get("end_date", END_DATE),
                external_tickers=external_tickers,
                external_feature_types=external_feature_types,
                base_df=base_df,
                base_feature_cols=base_feature_cols,
                close_col=close_col,
                save_to_excel=save_to_excel,
                experiment_dir=experiment_dir,
                skip_existing=skip_existing,
                run_name=f"optuna_holdout_trial_{trial.number:04d}",
                verbose=verbose,
            )

            score = compute_optuna_holdout_score(
                result=result,
                objective_metric=objective_metric,
                min_eval_count=min_eval_count,
                min_pred_1_count=min_pred_1_count,
            )

            row = {
                "trial_number": trial.number,
                "score": score,
                **cfg,
                objective_metric: get_holdout_metric_from_result(result, objective_metric),
                "test_eval_count": get_holdout_metric_from_result(result, "test_eval_count"),
                "test_pred_1_count": get_holdout_metric_from_result(result, "test_pred_1_count"),
                "skipped_existing": result.get("skipped_existing", False),
            }
            trial_rows.append(row)

            trial.set_user_attr("score", score)
            trial.set_user_attr(objective_metric, row[objective_metric])
            trial.set_user_attr("test_eval_count", row["test_eval_count"])
            trial.set_user_attr("test_pred_1_count", row["test_pred_1_count"])
            trial.set_user_attr("cfg", cfg)

            return score

        except Exception as e:
            if verbose:
                print(f"[OPTUNA HOLDOUT TRIAL FAIL] trial={trial.number}, error={e}")
            trial_rows.append({
                "trial_number": trial.number,
                "score": 0.0,
                **cfg,
                "error": str(e),
            })
            return 0.0

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame(trial_rows)
    if len(trials_df) > 0 and "score" in trials_df.columns:
        trials_df = trials_df.sort_values("score", ascending=False).reset_index(drop=True)

    if verbose:
        print("\n" + "=" * 80)
        print("OPTUNA HOLDOUT BEST TRIAL")
        print("=" * 80)
        print("best_value:", study.best_value)
        print("best_params:", study.best_params)
        display(trials_df.head(20))

    return {
        "study": study,
        "trials_df": trials_df,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "close_col": close_col,
    }


In [12]:

# =========================================================
# Holdout Optuna 실행 예시
# - 전체 데이터 마지막 3개월을 test로 사용
# - test 시작 전까지 train으로 변수 선택/학습
# - 결과: experiments_excel/experiment_summary_holdout.xlsx
# =========================================================

fixed_cfg = {
    "etf_code": "SMH",
    "test_months": 3,
    "vif_threshold": 10,
    "lag_search_years": 1,
    "lag_days": [1, 3, 5, 10, 20, 40, 60, 120],
    "random_state": 42,
    "n_rf_runs": 3,
    "n_repeats": 10,
}

holdout_optuna_result = run_optuna_holdout_experiments(
    n_trials=200,
    fixed_cfg=fixed_cfg,
    objective_metric="test_precision",
    min_eval_count=10,
    min_pred_1_count=3,
    save_to_excel=True,
    experiment_dir=EXCEL_EXPERIMENT_DIR,
    skip_existing=True,
    verbose=True,
)


Optuna holdout 공통 base_df 생성
Loading external ticker: QQQ / QQQ
Loading external ticker: SPY / SPY
Loading external ticker: SOXX / SOXX
Loading external ticker: NVDA / NVDA
Loading external ticker: TSM / TSM
Loading external ticker: VIX / ^VIX
Loading external ticker: TNX / ^TNX
Loading external ticker: USDKRW / KRW=X
Loading external ticker: DXY / DX-Y.NYB
Loading external ticker: GOLD / GC=F
Loading external ticker: OIL / CL=F


[I 2026-05-23 20:44:44,022] A new study created in memory with name: no-name-0b8948ef-d0eb-44e3-9abd-fb6c1c1d49d5



OPTUNA HOLDOUT TRIAL 1/200
{'etf_code': 'SMH', 'test_months': 3, 'vif_threshold': 10, 'lag_search_years': 1, 'lag_days': [1, 3, 5, 10, 20, 40, 60, 120], 'random_state': 42, 'n_rf_runs': 3, 'n_repeats': 10, 'start_date': '2020-01-01', 'end_date': None, 'n_days': 5, 'threshold': 0.01, 'top_n': 20, 'pred_threshold': 0.65, 'model_name': 'random_forest', 'model_params': {'n_estimators': 500, 'max_depth': 10, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'class_weight': 'balanced'}, 'search_method': 'optuna_holdout', 'optuna_trial_number': 0}
Holdout feature selection
train 기간: 2019-12-31 00:00:00 ~ 2026-02-20 00:00:00
test  기간: 2026-02-23 00:00:00 ~ 2026-05-22 00:00:00
train_base_df shape: (1543, 34)
test_base_df shape: (64, 34)
close_col: SMH_adj_close
VIF 계산 대상 row 수: 1484
VIF 계산 대상 feature 수: 32
현재 max VIF: 158.58 / feature: SMH_ret_20d
현재 max VIF: 121.48 / feature: SMH_ret_5d
현재 max VIF: 15.37 / feature: QQQ_ret_20d
현재 max VIF: 12.04 / feature: QQQ_ret_5d
현재 max VIF: 10.24 / feature:

,feature,importance_mean,importance_std,importance_var,importance_min,importance_max,run_count,repeat_count,importance_score,base_feature,selected_lag,best_lag,lag_corr,lag_abs_corr,lag_n_rows
0,DXY_ret_20d_lag120,0.123290,0.002489,6.197355e-06,0.119057,0.128485,3,30,0.120801,DXY_ret_20d,120,120,-0.222084,0.222084,132
1,USDKRW_ret_20d_lag1,0.077059,0.001727,2.982988e-06,0.074559,0.080329,3,30,0.075332,USDKRW_ret_20d,1,1,-0.239377,0.239377,251
2,GOLD_ret_5d_lag120,0.074402,0.002680,7.181190e-06,0.068122,0.079361,3,30,0.071722,GOLD_ret_5d,120,120,-0.178798,0.178798,132
3,TNX_level_lag120,0.072721,0.001518,2.305315e-06,0.069368,0.075558,3,30,0.071203,TNX_level,120,120,-0.186774,0.186774,132
4,OIL_ret_20d_lag60,0.061638,0.001838,3.379399e-06,0.057992,0.065463,3,30,0.059800,OIL_ret_20d,60,60,0.219382,0.219382,192
5,VIX_level_lag40,0.059311,0.001439,2.071372e-06,0.057185,0.062564,3,30,0.057872,VIX_level,40,40,0.163867,0.163867,212
6,TNX_diff_5d_lag20,0.059156,0.002600,6.759679e-06,0.054395,0.063171,3,30,0.056556,TNX_diff_5d,20,20,0.180109,0.180109,232
7,SMH_vol_20d_lag60,0.058574,0.002574,6.627958e-06,0.054366,0.062443,3,30,0.055999,SMH_vol_20d,60,60,-0.124807,0.124807,192
8,TSM_ret_20d_lag120,0.051865,0.001436,2.062305e-06,0.048627,0.053826,3,30,0.050429,TSM_ret_20d,120,120,-0.153156,0.153156,132
9,GOLD_ret_20d_lag1,0.051330,0.001163,1.352521e-06,0.049480,0.054350,3,30,0.050167,GOLD_ret_20d,1,1,0.135460,0.135460,251


6. valid_df 생성
top feature 수: 20
lag 생성 후 valid_df shape: (1607, 42)
원본 base_feature 제거 후 valid_df shape: (1607, 22)
최종 컬럼 수: 22
target 생성 후 valid_df shape: (1607, 24)
최종 valid_df shape: (64, 24)
valid_base_df shape: (64, 34)
valid_df 기간:
2026-02-23 00:00:00 ~ 2026-05-22 00:00:00
7. Train 학습 후 valid_df 예측
사용 feature 수: 20
train_df shape: (1398, 23)
X_train shape: (1398, 20)
train target 분포:
target_5d_up_1pct
0    720
1    678
Name: count, dtype: int64
target_5d_up_1pct
0    0.515021
1    0.484979
Name: proportion, dtype: float64
valid_df shape: (64, 24)
예측 가능한 valid row 수: 64
예측 불가능 row 수: 0
모델 학습 완료: random_forest
valid_df 예측 완료


...

예측값 분포:
pred
0    64
Name: count, dtype: Int64
8. 예측 1 기준 정확도 확인
평가 가능 row 수: 44
예측 1 개수: 0
예측 1 중 실제 1 개수: 0
예측 1 기준 정확도 precision: nan


,etf_code,n_days,threshold,lag_search_years,top_n,feature_count,model_name,model_params,eval_count,pred_1_count,...,accuracy,precision,recall,f1,auc,logloss,tn,fp,fn,tp
0,SMH,20,0.03,1,20,20,random_forest,"{'n_estimators': 300, 'max_depth': 7, 'min_sam...",44,0,...,0.227273,0.0,0.0,0.0,0.497059,0.686412,10,0,34,0


[I 2026-05-23 21:24:37,097] Trial 199 finished with value: 0.0 and parameters: {'n_days': 20, 'threshold': 0.03, 'top_n': 20, 'pred_threshold': 0.7, 'model_name': 'random_forest', 'rf_n_estimators': 300, 'rf_max_depth': 7, 'rf_min_samples_leaf': 1, 'rf_max_features': 'log2'}. Best is trial 6 with value: 1.0.



Excel 저장 완료
holdout summary 파일: experiments_excel/experiment_summary_holdout.xlsx

OPTUNA HOLDOUT BEST TRIAL
best_value: 1.0
best_params: {'n_days': 10, 'threshold': 0.03, 'top_n': 20, 'pred_threshold': 0.6000000000000001, 'model_name': 'gradient_boosting', 'gb_n_estimators': 300, 'gb_learning_rate': 0.01, 'gb_max_depth': 5}


,trial_number,score,etf_code,test_months,vif_threshold,lag_search_years,lag_days,random_state,n_rf_runs,n_repeats,...,top_n,pred_threshold,model_name,model_params,search_method,optuna_trial_number,test_precision,test_eval_count,test_pred_1_count,skipped_existing
0,174,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,hist_gradient_boosting,"{'max_iter': 500, 'learning_rate': 0.03, 'max_...",optuna_holdout,174,1.0,44,9,True
1,138,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,logistic,"{'C': 1.0, 'class_weight': 'balanced', 'max_it...",optuna_holdout,138,1.0,44,4,True
2,115,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,logistic,"{'C': 1.0, 'class_weight': 'balanced', 'max_it...",optuna_holdout,115,1.0,44,12,False
3,182,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.70,hist_gradient_boosting,"{'max_iter': 300, 'learning_rate': 0.03, 'max_...",optuna_holdout,182,1.0,44,8,True
4,181,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.70,hist_gradient_boosting,"{'max_iter': 300, 'learning_rate': 0.03, 'max_...",optuna_holdout,181,1.0,44,8,True
5,121,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,gradient_boosting,"{'n_estimators': 300, 'learning_rate': 0.03, '...",optuna_holdout,121,1.0,44,6,True
6,65,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,hist_gradient_boosting,"{'max_iter': 500, 'learning_rate': 0.03, 'max_...",optuna_holdout,65,1.0,44,9,False
7,64,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.70,hist_gradient_boosting,"{'max_iter': 300, 'learning_rate': 0.03, 'max_...",optuna_holdout,64,1.0,44,8,True
8,62,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,hist_gradient_boosting,"{'max_iter': 300, 'learning_rate': 0.03, 'max_...",optuna_holdout,62,1.0,44,9,True
9,61,1.0,SMH,3,10,1,"[1, 3, 5, 10, 20, 40, 60, 120]",42,3,10,...,20,0.65,hist_gradient_boosting,"{'max_iter': 300, 'learning_rate': 0.03, 'max_...",optuna_holdout,61,1.0,44,9,True
